## **Exploration of LBP Brain & Blood Bulk RNA-Seq Dataset**

### **Goal**: To generate a high quality inputs and outputs for my predictive models of brain expressions
#### ***Jolie Hoang | April 12, 2025***


### **Preliminary Setup**: 
First we load the necessary libraries, along with a set of utility functions.


In [ ]:
#module load R/4.3.0 #this works for biomart

In [ ]:
## WORKFLOW TO CREATE 
https://github.mountsinai.org/pages/Beckmann-lab/LBP_Blood-Brain/LBP-Blood-Brain-Quality-Control.html

#1. Log onto github. The html file will be host via git In bash,
cd /sc/arion/projects/mscic1/results/jolie/LBP/blood-brain
ml R
module load pandoc/2.6
module load git
git ls-remote https://github.mountsinai.org/Beckmann-lab/hello.git
cd /sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/
git clone https://github.mountsinai.org/Beckmann-lab/LBP_Blood-Brain.git
cd LBP_Blood-Brain # Now you’re inside the working directory linked to your GitHub repo

#2. Update the RmD file as needed, this is just FYI, dont need to run this
path = /sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/
name = LBP Blood-Brain Quality Control.Rmd

#3. Render the RmD file into a html in R, 
library(rmarkdown)
rmarkdown::render("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/LBP Blood-Brain Quality Control.Rmd")

#4. In bash, (this is a bit redundant, might be best to keep the RmD file and html file in the same place)
#cp /sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/LBP-Blood-Brain-Quality-Control.html /sc/arion/projects/mscic1/results/jolie/git_host/LBP_Blood-Brain/
#cd /sc/arion/projects/mscic1/results/jolie/git_host/LBP_Blood-Brain
git add LBP-Blood-Brain-Quality-Control.html
git commit -m "Update QC report"
git push

In [ ]:
rm(list=ls())
library(data.table)
library(ggplot2)
library(readxl)
library(biomaRt)
library(dplyr)
library(edgeR)
library(limma)
library(variancePartition)
library(matrixStats)

#We also set the random seed for reproducibility:
set.seed(2025)

### **Data Loading and Preprocessing**
Now we load the RNA-seq data sets, including its metadata and gene expression data sets.

In [ ]:
## Metadata, which has everything 
metadata <- readRDS("/sc/arion/projects/psychgen/lbp/data/RAW/rna/bulk/fromSema4/CompiledData/lbp_allBatches_RAPiD_BLOODandBRAIN_CovariatesTable_PlusCellTypes_530LIVINGSamples_14FEB2022.RDS")          
dim(metadata) #530 170 
metadata[1:5, 1:10] #there are 6 ID variables
#uniqueN(metadata[,.(IID_ISMMS)]) #there are 172 people total 
length(unique(metadata$IID_ISMMS)) #there are 172 people total 

## just checking whether blood and brain samples pairing is correct
sub <- metadata[,c("IID_ISMMS","SAMPLE_ISMMS","LIMS_SEMA4")] #"LBPSEMA4BLOOD395_0"  in "LIMS_SEMA4"
sub_blood <- sub[grepl("BLOOD", SAMPLE_ISMMS)] #243  3
sub_brain <- sub[grepl("BRAIN", SAMPLE_ISMMS)] #287  3

a <- merge(sub_blood, sub_brain, by="IID_ISMMS") #431   5

## based on previous deep dive in the sample space in blood-brain LBP_clean.sh, I've identified a mislabel; 
#this sample will later be dropped, see Sample Space Exploration
## there was one sample (PT-0117) that was mismatched (they have L_Brain and R_Blood so I'll changed it to L_Blood)
metadata[metadata$IID_ISMMS == "PT-0117", c("mymet_tissue", "mymet_brain", "mymet_timepoint")]
idx <- which(metadata$IID_ISMMS == "PT-0117" & metadata$mymet_tissue == "R_Blood")
metadata$mymet_tissue[idx] <- "L_Blood"
metadata$mymet_timepoint[idx] <- "left"
## Check again to see if the mislabeling is fixed
metadata[metadata$IID_ISMMS == "PT-0117", c("mymet_tissue", "mymet_brain", "mymet_timepoint")]

## How many blood and brain samples do I have?
table(metadata$mymet_brain)
# blood brain 
#   243   287 --> sum = 530
table(metadata$mymet_tissue)
# L_Blood L_Brain R_Blood R_Brain 
#     129     157     114     130 

brain_metadata <- metadata[metadata$mymet_brain == "brain", ]
dim(brain_metadata) #287 x 170 columns
#uniqueN(brain_metadata[,.(IID_ISMMS)]) #171 people with brain samples | this means that one person has only blood sample then 
length(unique(brain_metadata$IID_ISMMS)) #171 people with brain samples

blood_metadata <- metadata[metadata$mymet_brain == "blood", ]
dim(blood_metadata) #243 x 170
#unique(blood_metadata[,.(IID_ISMMS)]) #155 people with blood samples

blood_metadata <- as.data.frame(blood_metadata)
rownames(blood_metadata) <- blood_metadata$SAMPLE_ISMMS

brain_metadata <- as.data.frame(brain_metadata)
rownames(brain_metadata) <- brain_metadata$SAMPLE_ISMMS

In [ ]:
## Common samples, which have individuals who have BOTH blood and brain samples
common_samples <- merge(brain_metadata, blood_metadata, by = "IID_ISMMS")
dim(common_samples) #431 339
#uniqueN(common_samples[,.(IID_ISMMS)])  #there are 154 people with both blood and brain samples
length(unique(common_samples$IID_ISMMS))

colnames(common_samples) <- gsub("\\.x$", "_brain", colnames(common_samples))
colnames(common_samples) <- gsub("\\.y$", "_blood", colnames(common_samples))

tbl <- table(common_samples$IID_ISMMS)
sum(tbl == 1)  # number of IDs with 1 sample = 37
sum(tbl == 2)  # number of IDs with 2 samples = 37
sum(tbl == 4)  # number of IDs with 4 samples = 73 x4 | 80

## Deep dive into sample space
############################################
############################################
one_sample <- common_samples %>%
  group_by(IID_ISMMS) %>%
  filter(n() == 1)  
one_sample <- as.data.frame(one_sample)  
dim(one_sample) #37 339 | 37 people with only 1 sample

one_sample[,c("IID_ISMMS","mymet_tissue_blood","mymet_tissue_brain")]
table(one_sample$mymet_tissue_blood)
table(one_sample$mymet_tissue_brain)
#L_Blood L_Brain R_Blood R_Brain 
#      0      26       0      11 

##check whether mymet_tissue_brain and mymet_tissue_blood start with the same letter (either "R" or "L") for the same IID_ISMMS in one_sample
one_sample$same_prefix <- substr(one_sample$mymet_tissue_brain, 1, 1) == substr(one_sample$mymet_tissue_blood, 1, 1)

# Count TRUE vs FALSE cases
table(one_sample$same_prefix) ## ALL TRUE! Good.

one_sample$IID_ISMMS #PT-0019, 25, 140, 196

############################################
############################################
two_sample <- common_samples %>%
  group_by(IID_ISMMS) %>%
  filter(n() == 2)  
two_sample <- as.data.frame(two_sample)  
dim(two_sample) #74 339
table(two_sample$IID_ISMMS) #37 people with 2 samples | PT-0020, 32, 163

### 8 INDIVIDUALS WITH 1 BRAIN SAMPLE AND 2 BLOOD SAMPLES
# Count distinct tissue values per ID
id_counts <- aggregate(mymet_tissue_brain ~ IID_ISMMS, data = two_sample, function(x) length(unique(x)))
# Get IDs where mymet_tissue_brain has only one unique value
valid_ids <- id_counts$IID_ISMMS[id_counts$mymet_tissue_brain == 1]

# Subset original data
one_brain_two_blood_samples <- two_sample[two_sample$IID_ISMMS %in% valid_ids, ]
# make sure the L_Brain-L_Blood and R_Brain-R_Blood pairing is accurate:
paired_one_brain_two_blood_samples <- one_brain_two_blood_samples %>%
  filter((mymet_tissue_brain == "L_Brain" & mymet_tissue_blood == "L_Blood") |
         (mymet_tissue_brain == "R_Brain" & mymet_tissue_blood == "R_Blood"))

table(paired_one_brain_two_blood_samples$IID_ISMMS) #8 people
# PT-0032 PT-0046 PT-0054 PT-0071 PT-0100 PT-0116 PT-0124 PT-0177 
#       2       2       2       2       2       2       2       2

##investigation in pairing
sub <- subset(common_samples, IID_ISMMS == "PT-0032")
sub[,c("IID_ISMMS","mymet_tissue_blood","mymet_tissue_brain","RNASeqMetrics_MEDIAN_5PRIME_BIAS_blood","RNASeqMetrics_MEDIAN_5PRIME_BIAS_brain")]
dictionary2 <- common_samples[,c("IID_ISMMS","SAMPLE_ISMMS_blood","SAMPLE_ISMMS_brain","mymet_tissue_brain","mymet_tissue_blood")] #can also do this with blood ID 
"number_of_pair_brain"

dictionary2_matched <- dictionary2[dictionary2$mymet_tissue_brain == gsub("_Blood", "_Brain", dictionary2$mymet_tissue_blood), ]

#   IID_ISMMS mymet_tissue_blood mymet_tissue_brain
#45   PT-0032            L_Blood            L_Brain
#46   PT-0032            R_Blood            L_Brain
#   RNASeqMetrics_MEDIAN_5PRIME_BIAS_blood
#45                               1.025763
#46                               0.939081
#   RNASeqMetrics_MEDIAN_5PRIME_BIAS_brain
#45                               0.965238
#46                               0.965238

table(paired_one_brain_two_blood_samples$mymet_tissue_brain)
#L_Blood L_Brain R_Blood R_Brain 
#      0       6       0       2

### 29 INDIVIDUALS WITH 2 BRAIN SAMPLES AND 1 BLOOD SAMPLE
# Count distinct tissue values per ID
id_counts <- aggregate(mymet_tissue_brain ~ IID_ISMMS, data = two_sample, function(x) length(unique(x)))

# Get IDs where mymet_tissue_brain has only one unique value
valid_ids <- id_counts$IID_ISMMS[id_counts$mymet_tissue_brain == 2]

# Subset original data
two_brain_one_blood_sample <- two_sample[two_sample$IID_ISMMS %in% valid_ids, ]
# make sure the L_Brain-L_Blood and R_Brain-R_Blood pairing is accurate:
paired_two_brain_one_blood_sample <- two_brain_one_blood_sample %>%
  filter((mymet_tissue_brain == "L_Brain" & mymet_tissue_blood == "L_Blood") |
         (mymet_tissue_brain == "R_Brain" & mymet_tissue_blood == "R_Blood"))

table(two_brain_one_blood_sample$IID_ISMMS) #29 people
# PT-0020 PT-0056 PT-0084 PT-0088 PT-0091 PT-0094 PT-0102 PT-0109 PT-0111 PT-0115 
#       2       2       2       2       2       2       2       2       2       2 
# PT-0118 PT-0119 PT-0120 PT-0121 PT-0136 PT-0138 PT-0139 PT-0143 PT-0145 PT-0147 
#       2       2       2       2       2       2       2       2       2       2 
# PT-0154 PT-0157 PT-0161 PT-0163 PT-0172 PT-0174 PT-0175 PT-0179 PT-0184 
#       2       2       2       2       2       2       2       2       2
sub <- subset(common_samples, IID_ISMMS == "PT-0020")
sub[,c("IID_ISMMS","mymet_tissue_blood","mymet_tissue_brain","RNASeqMetrics_MEDIAN_5PRIME_BIAS_blood","RNASeqMetrics_MEDIAN_5PRIME_BIAS_brain")]
#  IID_ISMMS mymet_tissue_blood mymet_tissue_brain
#6   PT-0020            L_Blood            L_Brain
#7   PT-0020            L_Blood            R_Brain
#  RNASeqMetrics_MEDIAN_5PRIME_BIAS_blood RNASeqMetrics_MEDIAN_5PRIME_BIAS_brain
#6                               0.949469                               0.712104
#7                               0.949469                               0.599933

table(two_brain_one_blood_sample$mymet_tissue_blood)
#L_Blood L_Brain R_Blood R_Brain 
#     28       0      30       0

############################################
############################################
four_sample <- common_samples %>%
  group_by(IID_ISMMS) %>%
  filter(n() == 4)  
four_sample <- as.data.frame(four_sample)  

four_sample[1:5,1:5]

dim(four_sample) #320 339
table(four_sample$IID_ISMMS) #80 people 

##investigation in pairing
test2 <- subset(metadata, IID_ISMMS == "PT-0198")
test2[1:5,1:5]

test<- subset(four_sample, IID_ISMMS == "PT-0198")
test[1:5,1:5]

test[,c("IID_ISMMS","SAMPLE_ISMMS.x","mymet_extractiondate.x","mymet_tissue.x","SAMPLE_ISMMS.y","mymet_extractiondate.y","mymet_tissue.y")]

#LBPSEMA4BRAIN709 and LBPSEMA4BLOOD078 left
#LBPSEMA4BRAIN288 and LBPSEMA4BLOOD379 right

LBPSEMA4BRAIN709   PT-0198   LBPSEMA4BLOOD078   LBPSEMA4BRAIN709
LBPSEMA4BRAIN288   PT-0198   LBPSEMA4BLOOD379   LBPSEMA4BRAIN288

# make sure the L_Brain-L_Blood and R_Brain-R_Blood pairing is accurate:
paired_four_sample <- four_sample %>%
  filter((mymet_tissue_brain == "L_Brain" & mymet_tissue_blood == "L_Blood") |
         (mymet_tissue_brain == "R_Brain" & mymet_tissue_blood == "R_Blood"))
nrow(paired_four_sample) #160

one_sample$same_prefix <- NULL
two_sample$same_prefix <- NULL

############################################
############################################
common_samples_without_dup <- rbind(one_sample, paired_one_brain_two_blood_samples, paired_two_brain_one_blood_sample, paired_four_sample)
nrow(one_sample) #37
nrow(paired_one_brain_two_blood_samples) #8
nrow(paired_two_brain_one_blood_sample) #29
nrow(paired_four_sample) #160
dim(common_samples_without_dup) #234 339 (37 + 8 + 29 + 160 = 234)

##drop 1 sample with ID (IID_ISMMS == "PT-0117") because brain and blood sample with this ID might be collected from different time point
common_samples_without_dup <- common_samples_without_dup[common_samples_without_dup$IID_ISMMS != "PT-0117", ]
dim(common_samples_without_dup) #233 339

length(unique(common_samples_without_dup$IID_ISMMS)) #153 people

brain_metadata <-common_samples_without_dup[common_samples_without_dup$mymet_brain_brain == "brain", ] 
blood_metadata <-common_samples_without_dup[common_samples_without_dup$mymet_brain_blood == "blood", ] 

identical(brain_metadata$SAMPLE_ISMMS_brain, blood_metadata$SAMPLE_ISMMS_brain) #TRUE
identical(brain_metadata$SAMPLE_ISMMS_blood, blood_metadata$SAMPLE_ISMMS_blood) #TRUE

##good sanity check here: 
dim(blood_metadata) #233 339
dim(brain_metadata) #233 339
identical(brain_metadata, blood_metadata) #TRUE | this is because I the columns are "duplicated" with underscore
#_blood and _brain

length(unique(blood_metadata$IID_ISMMS)) #153
length(unique(brain_metadata$IID_ISMMS)) #153


In [ ]:
##Expression data
raw_count <- readRDS("/sc/arion/projects/psychgen/lbp/data/RAW/rna/bulk/fromSema4/CompiledData/lbp_allBatches_RAPiD_featureCounts_Compiled_BLOODandBRAIN_530LIVINGsamples_15FEB2022.RDS")                                                                                   
dim(raw_count) ##58929 x 530  

blood_ge <- raw_count[, colnames(raw_count) %in% blood_metadata$SAMPLE_ISMMS_blood]
dim(blood_ge) #58929 x 233

brain_ge <- raw_count[, colnames(raw_count) %in% brain_metadata$SAMPLE_ISMMS_brain]
dim(brain_ge) #58929 x 233

identical(colnames(blood_ge), blood_metadata$SAMPLE_ISMMS_blood) #FALSE
# Reorder columns of brain_ge to match the order of brain_metadata$SAMPLE_ISMMS_brain
blood_ge <- blood_ge[, match(blood_metadata$SAMPLE_ISMMS_blood, colnames(blood_ge))]
identical(colnames(blood_ge), blood_metadata$SAMPLE_ISMMS_blood) #TRUE
rownames(blood_metadata) <- blood_metadata$SAMPLE_ISMMS_blood
identical(colnames(blood_ge),rownames(blood_metadata)) #TRUE

identical(colnames(brain_ge), brain_metadata$SAMPLE_ISMMS_brain) #FALSE
# Reorder columns of brain_ge to match the order of brain_metadata$SAMPLE_ISMMS_brain
brain_ge <- brain_ge[, match(brain_metadata$SAMPLE_ISMMS_brain, colnames(brain_ge))]
identical(colnames(brain_ge), brain_metadata$SAMPLE_ISMMS_brain) #TRUE
rownames(brain_metadata) <- brain_metadata$SAMPLE_ISMMS_brain
identical(rownames(brain_metadata), colnames(brain_ge)) #TRUE

save.image("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood-brain_sample_baseline_20250522.RData")

## CELL TYPE DECONVOLUTION

In [ ]:
# PATH OF RAPID - to get gene length info
#old path that doesn't exist anymore, this path is from 
blood_metadata$fastqDir
/sc/arion/projects/psychgen2/lbp/data/RAW/rna/bulk/fromSema4/Merged_Batches/200708_A00732_0077_AH7WMHDSXY.200708_A00732_0078_BH7YHKDSXY.Merged/LBPSEMA4BLOOD009
#this are the up-to-date paths as of May 7 2025
/sc/arion/projects/mscic1/lbp/data/bulkRNAseq/fromSema4/NPP536/Sample_ISM229910-2/RAPiD/featureCounts/Sample_ISM229910-2.primary.txt
/sc/arion/projects/mscic1/lbp/data/bulkRNAseq/fromSema4/190703_A00732_0026_AHML5TDSXX

In [ ]:
## extracting gene length from Sample_ISM229910-2.primary.txt
# 1. Read the featureCounts file
fc <- read.delim("/sc/arion/projects/mscic1/lbp/data/bulkRNAseq/fromSema4/NPP536/Sample_ISM229910-2/RAPiD/featureCounts/Sample_ISM229910-2.primary.txt",
                 comment.char = "#",  # ignore metadata header lines
                 stringsAsFactors = FALSE)

# 2. Extract Geneid and Length columns
gene_lengths_df <- fc[, c("Geneid", "Length")]

# 3. Match gene lengths to gene expression matrix
gene_lengths <- gene_lengths_df$Length[match(rownames(blood_ge), gene_lengths_df$Geneid)]

matched_ids <- gene_lengths_df$Geneid[match(rownames(blood_ge), gene_lengths_df$Geneid)]
all.equal(rownames(blood_ge), matched_ids) #TRUE

# 6. Add it as a column to blood_metadata (if blood_metadata is sample-level, this might not be appropriate)
# Instead, create a named vector or dataframe for gene-level info:
blood_ge_with_length <- cbind(blood_ge, gene_length = gene_lengths)

In [ ]:
/sc/arion/projects/mscic1/lbp/data/bulkRNAseq/fromSema4/NPP536/Sample_ISM223618-2/RAPiD/featureCounts/Sample_ISM223618-2.primary.txt

Geneid                  Chr     Start   End     Strand  Length  Sample_ISM223618-2.no_filter.bam
ENSG00000223972.5       chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1    11869;12010;12179;12613;12613;12975;13221;13221;13453   12227;120>
ENSG00000227232.5       chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1  14404;15005;15796;16607;16858;17233;17606;17915;18268;247>
ENSG00000278267.1       chr1    17369   17436   -       68      20
ENSG00000243485.5       chr1;chr1;chr1;chr1;chr1        29554;30267;30564;30976;30976   30039;30667;30667;31109;31097   +;+;+;+;+       1>
ENSG00000284332.1       chr1    30366   30503   +       138     0
ENSG00000237613.2       chr1;chr1;chr1;chr1;chr1        34554;35245;35277;35721;35721   35174;35481;35481;36073;36081   -;-;-;-;-       1>
ENSG00000268020.3       chr1    52473   53312   +       840     0
ENSG00000240361.2       chr1;chr1;chr1;chr1     57598;58700;62916;62949 57653;58856;64116;63887 +;+;+;+ 1414    0
ENSG00000186092.6       chr1;chr1;chr1;chr1     65419;65520;69037;69055 65433;65573;71585;70108 +;+;+;+ 2618    0
ENSG00000238009.6       chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1;chr1    89295;92091;92230;110953;>
ENSG00000239945.1       chr1;chr1       89551;90287     90050;91105     -;-     1319    1
ENSG00000233750.3       chr1    131025  134836  +       3812    0
ENSG00000268903.1       chr1    135141  135895  -       755     0
ENSG00000269981.1       chr1    137682  137965  -       284     3
ENSG00000239906.1       chr1;chr1       139790;140075   139847;140339   -;-     323     1

In [ ]:
## TPM conversion
# Step 1: Separate gene lengths and raw counts
gene_lengths_named <- setNames(gene_lengths_df$Length, gene_lengths_df$Geneid)
gene_length_kb <- gene_lengths_named[rownames(blood_ge)] / 1000

# Step 2: Calculate RPK, reads per kilobase
rpk <- sweep(blood_ge, 1, gene_length_kb, FUN = "/")

# Step 3: Calculate TPM
tpm <- sweep(rpk, 2, colSums(rpk), FUN = "/") * 1e6
dim(tpm) #58929 x 243

## Emsembl ID conversion to gene symbols
# Connect to Ensembl
ensembl <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")

# Strip versions from Ensembl IDs
ensembl_ids <- sub("\\..*", "", rownames(tpm))  # Remove version -- does this matter? it might, ignore this for now

# Map to HGNC symbols
id_map <- getBM(
  attributes = c("ensembl_gene_id", "hgnc_symbol"),
  filters = "ensembl_gene_id",
  values = ensembl_ids,
  mart = ensembl
)

# Remove duplicates and empty symbols
id_map <- id_map[id_map$hgnc_symbol != "", ]
id_map <- id_map[!duplicated(id_map$hgnc_symbol), ]

dim(id_map) #57597 x 2; 1332 genes with no hgnc_symbol

# Match and filter
tpm$ensembl_id <- sub("\\..*", "", rownames(tpm))
tpm_mapped <- merge(tpm, id_map, by.x = "ensembl_id", by.y = "ensembl_gene_id")
tpm_final <- tpm_mapped[, c("hgnc_symbol", colnames(tpm)[1:(ncol(tpm)-1)])]
rownames(tpm_final) <- tpm_final$hgnc_symbol
tpm_final <- tpm_final[, -1]  # remove hgnc_symbol column (it's now rownames)


In [ ]:
## TPM conversion using Ensemble IDs -- OLD VERSION 
# Replace gene_ids with your actual gene symbols or Ensembl IDs
gene_ids <- rownames(raw_count)
gene_ids <- sub("\\..*", "", gene_ids) #remove numbers after decimal

# Function to get gene lengths from Ensembl
get_gene_lengths <- function(gene_ids, id_type = "ensembl_gene_id") {
  # Connect to Ensembl
  ensembl <- useEnsembl(biomart = "genes", dataset = "hsapiens_gene_ensembl") #check version
  #cat("Using Ensembl version:", listMarts(ensembl)[1,2], "\n")
  # Retrieve gene lengths (using transcript length as proxy)
  gene_info <- getBM(
    attributes = c(id_type, "hgnc_symbol", "gene_biotype", "transcript_length"),
    filters = id_type,
    values = gene_ids,
    mart = ensembl
  )
  
  # Aggregate to get the longest transcript for each gene
  # (common practice for TPM calculation)
  gene_lengths <- gene_info %>%
    group_by(across(all_of(id_type)), hgnc_symbol, gene_biotype) %>%
    summarize(length = max(transcript_length, na.rm = TRUE), .groups = "drop")
  
  return(gene_lengths)
}

gene_lengths_df <- get_gene_lengths(gene_ids, id_type = "ensembl_gene_id")
dim(gene_lengths_df) #57597 x  4 ##seems like there are genes with no gene length?

write.csv(gene_lengths_df, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_gene_lengths.csv", row.names = FALSE)

# Convert to a named vector for easier use in TPM calculation
gene_lengths <- gene_lengths_df$length
names(gene_lengths) <- gene_lengths_df$ensembl_gene_id

##convert to TPM
library(DGEobj.utils)

# Try converting to a numeric matrix first
raw_count_matrix <- as.matrix(raw_count)
mode(raw_count_matrix) <- "numeric"
str(raw_count_matrix)

dim(raw_count_matrix) #58929 x 530
length(gene_lengths) #57597

# Remove decimal points and numbers after them in rownames
rownames(raw_count_matrix) <- gsub("\\.\\d+$", "", rownames(raw_count_matrix))
head(raw_count_matrix)

# Get only the genes that have length information
common_genes <- intersect(rownames(raw_count_matrix), names(gene_lengths))
length(common_genes)  # This will show how many genes overlap

# Subset both objects to include only common genes
raw_count_sub <- raw_count_matrix[common_genes, ]
gene_lengths_sub <- gene_lengths[common_genes] #not sure why 3 genes were removed here

tpm_matrix <- convertCounts(
  countsMatrix = raw_count_sub,  # Your raw count matrix
  unit = "TPM",               # Target unit (TPM)
  geneLength = gene_lengths_sub,  # Gene lengths vector you got from Ensembl
  log = FALSE,                # Don't log-transform the data
  normalize = "none",         # No additional normalization
  prior.count = 0             # No prior count added (can help with zeros)
)

# Check the result
head(tpm_matrix)
write.csv(tpm_matrix, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_tpm_matrix.csv", row.names = TRUE)
test<-read.csv("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_tpm_matrix.csv",row.names = TRUE)

write.table(tpm_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_tpm_matrix.txt", sep = "\t", quote = FALSE, col.names = NA)
test2<-fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_tpm_matrix.txt",data.table=FALSE)

# Create a lookup table: Ensembl ID → HGNC symbol
id_to_symbol <- gene_lengths_df %>%
  filter(ensembl_gene_id %in% rownames(tpm_matrix)) %>%
  select(ensembl_gene_id, hgnc_symbol) %>%
  distinct() %>%
  filter(!is.na(hgnc_symbol), hgnc_symbol != "")

# Keep only unique mappings
id_to_symbol <- id_to_symbol[!duplicated(id_to_symbol$ensembl_gene_id), ]
id_to_symbol <- as.data.frame(id_to_symbol)
rownames(id_to_symbol) <- id_to_symbol$ensembl_gene_id

# Replace rownames in tpm_matrix
tpm_matrix_renamed <- tpm_matrix[rownames(tpm_matrix) %in% id_to_symbol$ensembl_gene_id, ]
rownames(tpm_matrix_renamed) <- id_to_symbol[rownames(tpm_matrix_renamed), "hgnc_symbol"]

# Optional: remove duplicates in HGNC symbols (some symbols map to multiple Ensembl IDs)
tpm_matrix_final <- tpm_matrix_renamed[!duplicated(rownames(tpm_matrix_renamed)), ] ## what to do about this


### Decision Points: 
1. within each sample in /sc/arion/projects/mscic1/lbp/data/bulkRNAseq/fromSema4/NPP536/Sample_ISM229910-2/, there are: bams  fastqc  featureCounts  kallisto  leafcutter  qc_metrics  rsem  star
right now i'm using featureCounts, but be aware of other options (what are they really) like kallisto, leafcutter, etc.
2. within featureCounts, there are 
Sample_ISM229912-2.exon.geneID.txt
Sample_ISM229912-2.exon.geneID.txt.summary
Sample_ISM229912-2.exon.transcriptID.txt
Sample_ISM229912-2.exon.transcriptID.txt.summary
Sample_ISM229912-2.exon.txt
Sample_ISM229912-2.exon.txt.summary
Sample_ISM229912-2.primary.txt
Sample_ISM229912-2.primary.txt.summary
--

the main feature counts we normally use is derived from Sample_ISM229912-2.primary.txt
but antoher file (e.g. Sample_ISM229912-2.exon.geneID.txt) might be better (know the differences at a global level)

## Exploring metadata
174 columns with estimated numbers of column in (): 
- ID variables (6): SAMPLE_ISMMS, ISM_SEMA4, RSM_SEMA4, LIMS_SEMA4, BARCODE_ISMMS, IID_ISMMS
- FASTQ variables (6): FASTQC_Adapter_Content,etc.
- FEATURECOUNTS variables (4): FEATURECOUNTS_Assigned, etc.
- STAR variables (21): STAR_Average_mapped_length, etc.
- AlignmentSummaryMetrics variables (43): AlignmentSummaryMetrics_TOTAL_READS_PAIR, etc.
- InsertSizeMetrics variables (18): InsertSizeMetrics_MEDIAN_INSERT_SIZE, etc.
- RNASeqMetrics variables (22): RNASeqMetrics_PF_BASES, etc.
- GcBiasMetrics variables (9): GcBiasMetrics_TOTAL_CLUSTERS, etc.
- Cell type counts (16): Lymphocyte_total_lm22, Monocytes_lm22, Mono_Macro_DC_lm22, etc.
- File info (3): rapidBatch, s4newbatch, fastqDir
- My metadata info (18): mymet_PLATE, mymet_extractionbatch, mymet_seqbatch, mymet_depletionbatch, mymet_rin, bbstatus, etc.


In [ ]:
#REMOVING SAMPLES WITH LOW RIN 

#summary(metadata$mymet_rin)
#Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
#4.100   6.700   7.400   7.333   8.000  10.000 

#boxplot of tissues by RIN
#plot<- ggplot(metadata, aes(x = mymet_tissue, y = mymet_rin)) +
#  geom_boxplot() +
#  xlab("Tissue") +
#  ylab("RIN") +
#  ggtitle("Boxplot of RIN by Tissue") +
#  theme_minimal()

##blood RIN is higher than brain RIN!
#ggsave(filename = "/hpc/users/hoangd02/www/plots/rin_by_tissue.png",
#        plot = plot, 
#        width = 6, height = 6, dpi = 300)

#The quality metric shown is RIN (RNA Integrity Number, as measured by Agilent Bioanalyzer). 
#All samples with a RIN of 6.0 or higher qualify for RNA Sequence analysis.
#https://www.gtexportal.org/home/methods

#metadata_sub <- metadata[metadata$mymet_rin >= 6, ]
#dim(metadata_sub) #479 x 170

##REMOVED 530 - 479 = 51 SAMPLES
#table(metadata_sub$mymet_brain)
# blood brain 
#   214   265
#table(metadata_sub$mymet_tissue)
# L_Blood L_Brain R_Blood R_Brain 
#     115     145     99     120

#brain_metadata <-metadata_sub[metadata_sub$mymet_brain == "brain", ]
#dim: 265 x 170 
#blood_metadata <-metadata_sub[metadata_sub$mymet_brain == "blood", ] 
#dim: 214 x 170


Decision point: 
1. Remove globin genes for blood/ brain/ both?

First QC attempt: remove globin genes in both tissues

In [ ]:
## globin genes removal - manually mapping because biomaRt SUCKS
#https://useast.ensembl.org/Homo_sapiens/Gene/Summary?db=core;g=ENSG00000161544;r=17:76527356-76551175
globin_genes <- c("CYGB", "HBA1", "HBA2", 
                  "HBB", "HBD", "HBE1", 
                  "HBG1", "HBG2", "HBM", 
                  "HBQ1", "HBZ", "MB") #12 genes

globin_ensembl_ids <- c("ENSG00000161544.10", "ENSG00000206172.8", "ENSG00000188536.13", 
                         "ENSG00000244734.4", "ENSG00000223609.11", "ENSG00000213931.7", 
                         "ENSG00000213934.9", "ENSG00000196565.15", "ENSG00000206177.7", 
                         "ENSG00000086506.3", "ENSG00000130656.6", "ENSG00000198125.13")
#12 genes
#check version (numbers after the decimal)

#grab rows where row names match the IDs in globin_ensembl_ids
matched_rows <- blood_ge[rownames(blood_ge) %in% globin_ensembl_ids, ]
matched_rows #seems like we have 9 globin genes in our data

#there are some globin genes in brain too, remove
matched_rows <- brain_ge[rownames(brain_ge) %in% globin_ensembl_ids, ]
matched_rows #seems like we have 9 globin genes in our data

#which of 12 globin genes don't I have
missing_genes <- setdiff(globin_ensembl_ids, rownames(matched_rows))
missing_genes
#we dont have "ENSG00000223609.11" "ENSG00000213934.9"  "ENSG00000130656.6"
#aka "HBD", "HBG1" or "HBZ"

## gene expression is pretty different for samples -- highlighting the importance of normalization 
#summary(blood_ge$LBPSEMA4BLOOD792)
#     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
#      0.0       1.0       6.0     290.7      43.0 1799229.0 
#summary(blood_ge$LBPSEMA4BLOOD794)
#   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
#      0       1       6     613      55 6216919

#subset globin rows to see actual raw amount 
globin_rows_blood <- blood_ge[rownames(blood_ge) %in% globin_ensembl_ids, ] #max = in the thousands
globin_rows_brain <- brain_ge[rownames(brain_ge) %in% globin_ensembl_ids, ] # not 0, max = in the hundreds

# Remove globin genes from blood_ge
dim(blood_ge) #58929   233 (all samples: 243)
blood_ge <- blood_ge[!rownames(blood_ge) %in% globin_ensembl_ids, ]
dim(blood_ge) #58920   233

# Remove globin genes from brain_ge -- check the actual amount -- remove 
dim(brain_ge) #58929   233 (all samples: 287)
brain_ge <- brain_ge[!rownames(brain_ge) %in% globin_ensembl_ids, ]
dim(brain_ge) #58920   233

In [ ]:
######################## IGNORE FOR NOW 
ensembl <- useEnsembl(biomart = "genes", dataset = "hsapiens_gene_ensembl") # need to use the right reference here
gene_ids_brain <- rownames(v_brain$E) #21356 225
gene_ids_brain <- sub("\\..*", "", gene_ids_brain)
gene_ids_blood <- rownames(v_blood$E)
gene_ids_blood <- sub("\\..*", "", gene_ids_blood)

gene_info_brain <- getBM(
  attributes = c("ensembl_gene_id", "hgnc_symbol", "gene_biotype"),
  filters = "ensembl_gene_id",
  values = gene_ids_brain,
  mart = ensembl
)
dim(gene_info_brain) #20970     3
head(gene_info_brain)
#  ensembl_gene_id hgnc_symbol   gene_biotype
#1 ENSG00000000003      TSPAN6 protein_coding

length(unique(gene_info_brain$hgnc_symbol)) #18190 - there are A LOT of duplicates for hgnc_symbol
length(unique(gene_info_brain$ensembl_gene_id)) #20968 - there are 2 duplicates for ensembl_gene_id

##What to do when Ensembl gene IDs could not be matched in Ensembl BioMart? maybe use a different version of useEnsembl()?

table(gene_info_brain$gene_biotype)
                          artifact                          IG_V_gene 
                                 6                                  2 
                            lncRNA                              miRNA 
                              3177                                204 
                          misc_RNA                            Mt_rRNA 
                               186                                  2 
                           Mt_tRNA               processed_pseudogene 
                                22                                672 
                    protein_coding                         pseudogene 
                             15184                                  1 
                          ribozyme                    rRNA_pseudogene 
                                 2                                  3 
                            scaRNA                             snoRNA 
                                13                                278 
                             snRNA                               sRNA 
                                88                                  1 
                               TEC                          TR_C_gene 
                               304                                  3 
                   TR_V_pseudogene   transcribed_processed_pseudogene 
                                 1                                204 
    transcribed_unitary_pseudogene transcribed_unprocessed_pseudogene 
                                55                                499 
                unitary_pseudogene             unprocessed_pseudogene 
                                 1                                 61 
                         vault_RNA 
                                 1 

gene_info_brain$category <- dplyr::case_when(
  gene_info_brain$gene_biotype == "protein_coding" ~ "Protein Coding",
  gene_info_brain$gene_biotype %in% c("lncRNA", "processed_transcript", "TEC") ~ "lncRNA",
  gene_info_brain$gene_biotype %in% c("miRNA", "snoRNA", "snRNA", "scaRNA", "sRNA", "vault_RNA", "misc_RNA", "ribozyme") ~ "miRNA / sncRNA",
  gene_info_brain$gene_biotype %in% c(
    "processed_pseudogene", "transcribed_processed_pseudogene", "transcribed_unitary_pseudogene",
    "transcribed_unprocessed_pseudogene", "unitary_pseudogene", "unprocessed_pseudogene",
    "pseudogene", "rRNA_pseudogene", "TR_V_pseudogene"
  ) ~ "Pseudogene",
  gene_info_brain$gene_biotype %in% c("Mt_rRNA", "Mt_tRNA") ~ "Mitochondrial RNA",
  gene_info_brain$gene_biotype %in% c("IG_V_gene", "TR_C_gene") ~ "Immune-Related",
  TRUE ~ "Other / Artifact"
)

table(gene_info_brain$category)
   Immune-Related            lncRNA    miRNA / sncRNA Mitochondrial RNA 
                5              3481               773                24 
 Other / Artifact    Protein Coding        Pseudogene 
                6             15184              1497 

##BRAIN: 
#lncRNA = a non-coding gene/transcript >200bp in length = 3177
#ncRNA = a non-coding gene 
#pseudogene = #A gene that has homology to known protein-coding genes but contain a frameshift and/or stop codon(s) which disrupts the ORF. Thought to have arisen through duplication followed by loss of function.
#other = IG_V_gene (2)  
#what to do with artifact?

################## BLOOD ##################
###########################################
gene_info_blood <- getBM(
  attributes = c("ensembl_gene_id", "hgnc_symbol", "gene_biotype"),
  filters = "ensembl_gene_id",
  values = gene_ids_blood,
  mart = ensembl
)
dim(gene_info_blood) #20970     3
head(gene_info_blood)
#  ensembl_gene_id hgnc_symbol   gene_biotype
#1 ENSG00000000003      TSPAN6 protein_coding

length(unique(gene_info_blood$hgnc_symbol)) #18333 - there are A LOT of duplicates for hgnc_symbol
length(unique(gene_info_blood$ensembl_gene_id)) #20759 - there are 2 duplicates for ensembl_gene_id

table(gene_info_blood$gene_biotype)
                          artifact                          IG_C_gene 
                                 4                                 14 
                   IG_C_pseudogene                          IG_J_gene 
                                 2                                 12 
                   IG_J_pseudogene                          IG_V_gene 
                                 1                                 68 
                   IG_V_pseudogene                             lncRNA 
                                 3                               2796 
                             miRNA                           misc_RNA 
                               201                                175 
                           Mt_rRNA                            Mt_tRNA 
                                 2                                 22 
              processed_pseudogene                     protein_coding 
                               518                              15436 
                          ribozyme                               rRNA 
                                 2                                  1 
                   rRNA_pseudogene                             scaRNA 
                                 4                                 17 
                            snoRNA                              snRNA 
                               201                                 89 
                              sRNA                                TEC 
                                 1                                262 
                         TR_C_gene                          TR_D_gene 
                                 6                                  1 
                         TR_J_gene                    TR_J_pseudogene 
                                76                                  2 
                         TR_V_gene                    TR_V_pseudogene 
                                84                                  3 
  transcribed_processed_pseudogene     transcribed_unitary_pseudogene 
                               152                                 70 
transcribed_unprocessed_pseudogene                 unitary_pseudogene 
                               463                                  5 
            unprocessed_pseudogene                          vault_RNA 
                                68                                  1

gene_info_blood$category <- dplyr::case_when(
  gene_info_blood$gene_biotype == "protein_coding" ~ "Protein Coding",
  gene_info_blood$gene_biotype %in% c("lncRNA", "TEC") ~ "lncRNA",
  gene_info_blood$gene_biotype %in% c(
    "miRNA", "snoRNA", "snRNA", "scaRNA", "sRNA",
    "vault_RNA", "misc_RNA", "ribozyme"
  ) ~ "miRNA / sncRNA",
  gene_info_blood$gene_biotype %in% c(
    "processed_pseudogene", "transcribed_processed_pseudogene",
    "transcribed_unitary_pseudogene", "transcribed_unprocessed_pseudogene",
    "unitary_pseudogene", "unprocessed_pseudogene", "rRNA_pseudogene",
    "IG_C_pseudogene", "IG_J_pseudogene", "IG_V_pseudogene",
    "TR_J_pseudogene", "TR_V_pseudogene"
  ) ~ "Pseudogene",
  gene_info_blood$gene_biotype %in% c("Mt_rRNA", "Mt_tRNA") ~ "Mitochondrial RNA",
  gene_info_blood$gene_biotype == "rRNA" ~ "rRNA",
  gene_info_blood$gene_biotype %in% c(
    "IG_C_gene", "IG_J_gene", "IG_V_gene",
    "TR_C_gene", "TR_D_gene", "TR_J_gene", "TR_V_gene"
  ) ~ "Immune-Related",
  TRUE ~ "Artifact / Other"
)

table(gene_info_blood$category)
# Artifact / Other    Immune-Related            lncRNA    miRNA / sncRNA 
#                4               261              3058               687 
#Mitochondrial RNA    Protein Coding        Pseudogene              rRNA 
#               24             15436              1291                 1 

table(gene_info_brain$category)
#   Immune-Related            lncRNA    miRNA / sncRNA Mitochondrial RNA 
#                5              3481               773                24 
# Other / Artifact    Protein Coding        Pseudogene 
#                 6             15184              1497 

common_ids <- intersect(gene_info_blood$ensembl_gene_id, gene_info_brain$ensembl_gene_id)
length(common_ids) #17322 genes
only_in_blood <- setdiff(gene_info_blood$ensembl_gene_id, gene_info_brain$ensembl_gene_id)
length(only_in_blood) #3437
only_in_brain <- setdiff(gene_info_brain$ensembl_gene_id, gene_info_blood$ensembl_gene_id)
length(only_in_brain) #3646

# Common genes df
common_blood <- gene_info_blood[gene_info_blood$ensembl_gene_id %in% common_ids, ]
common_brain <- gene_info_brain[gene_info_brain$ensembl_gene_id %in% common_ids, ]

# Unique genes
unique_blood <- gene_info_blood[gene_info_blood$ensembl_gene_id %in% only_in_blood, ] #3438    4 - 1 dup here
unique_brain <- gene_info_brain[gene_info_brain$ensembl_gene_id %in% only_in_brain, ] #3646    4

##finding dup
## BLOOD
duplicated_ids <- common_blood$ensembl_gene_id[duplicated(common_blood$ensembl_gene_id)]
unique(duplicated_ids) #"ENSG00000230417" "ENSG00000280739"
common_blood[common_blood$ensembl_gene_id %in% c("ENSG00000230417", "ENSG00000280739"), ]
      ensembl_gene_id hgnc_symbol gene_biotype category
15938 ENSG00000230417   LINC00595       lncRNA   lncRNA
15939 ENSG00000230417   LINC00856       lncRNA   lncRNA
20226 ENSG00000280739   EIF1B-AS1       lncRNA   lncRNA
20227 ENSG00000280739  ENTPD3-AS1       lncRNA   lncRNA

## BRAIN -- same as brain 
duplicated_ids <- common_brain$ensembl_gene_id[duplicated(common_brain$ensembl_gene_id)]
unique(duplicated_ids) #"ENSG00000230417" "ENSG00000280739"
common_brain[common_brain$ensembl_gene_id %in% c("ENSG00000230417", "ENSG00000280739"), ]

# Biotype/category distribution
table(common_blood$category) #this is the same as table(common_brain$category)
# Artifact / Other    Immune-Related            lncRNA    miRNA / sncRNA 
#                4                 5              2025               440 
#Mitochondrial RNA    Protein Coding        Pseudogene 
#               24             13986               840

table(unique_blood$category)
#Immune-Related         lncRNA miRNA / sncRNA Protein Coding     Pseudogene 
#           256           1033            247           1450            451 
#          rRNA 
#             1
table(unique_brain$category)
#          lncRNA   miRNA / sncRNA Other / Artifact   Protein Coding 
#            1456              333                2             1198 
#      Pseudogene 
#             657 



In [ ]:
####### IGNORE FOR NOW, OLD CODE
### BRAIN, following Lora's paper 
# Convert raw counts to DGEList
dge_brain <- DGEList(counts = brain_ge)

# Convert to CPM
cpm_brain <- cpm(dge_brain)

# Filtering: Keep genes with CPM >= 1 in at least 10% of samples -- another decision point 
min_samples <- ceiling(0.1 * ncol(brain_ge))  # 10% of samples
keep_genes <- rowSums(cpm_brain >= 1) >= min_samples
dge_brain <- dge_brain[keep_genes, , keep.lib.sizes = FALSE]  # Apply filtering

# Apply TMM normalization for composition bias
dge_brain <- calcNormFactors(dge_brain, method = "TMM")

vobj <- voom(dge_brain, plot = TRUE) ### THIS LOOKS GOOD

# Define formula 
form <- ~ mymet_rin + (1|IID_ISMMS) + (1|mymet_sex)
varPart <- fitExtractVarPartModel(vobj, form, brain_metadata)

plot <- plotVarPart(varPart) + theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Proportion of Variance Explained by Each Covariate")

ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/brain_variance_partition_all_samples.png",
       plot = plot)

https://hoangd02.u.hpc.mssm.edu/plots/lbp/brain_variance_partition_all_samples.png

#this is before samples with RIN >= 6
https://hoangd02.u.hpc.mssm.edu/plots/lbp/brain_variance_partition.png

# Fit linear mixed model
fit <- dream(vobj, ~ mymet_rin + (1|IID_ISMMS) + (1|mymet_sex), brain_metadata)
#Total:1230 s
#Warning message:
#In dream(vobj, ~mymet_rin + (1 | IID_ISMMS) + (1 | mymet_sex), brain_metadata) :
#  Sample names of responses (i.e. columns of exprObj) do not match
#sample names of metadata (i.e. rows of data).  Recommend consistent
#names so downstream results are labeled consistently.

# Get residuals
brain_resid_matrix <- residuals(fit)
dim(brain_resid_matrix)
#21296   265

write.table(brain_resid_matrix, 
            file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_resid_matrix_all_samples.txt", 
            sep = "\t",            # tab-delimited
            quote = FALSE,         # don't put quotes around values
            row.names = TRUE,      # keep gene names as rownames
            col.names = NA)        # keep sample names as column headers

#test <- fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_resid_matrix.txt",data.table = FALSE)
#test$V1 is gene names

##PCA on residualized matrix
# Center the residual matrix (genes x samples)
# If rows = genes, columns = samples
resid_matrix_t <- t(brain_resid_matrix)  # prcomp expects samples as rows

# Perform PCA
pca_res <- prcomp(resid_matrix_t, center = TRUE, scale. = TRUE)

# View variance explained
summary(pca_res)
#Importance of components:
#                           PC1     PC2      PC3     PC4      PC5      PC6
#Standard deviation     82.9080 51.6385 43.22457 33.1676 29.43829 26.19070
#Proportion of Variance  0.3224  0.1251  0.08763  0.0516  0.04065  0.03217
#Cumulative Proportion   0.3224  0.4475  0.53509  0.5867  0.62733  0.65950
#                            PC7     PC8      PC9     PC10     PC11     PC12
#Standard deviation     22.31250 19.9148 17.64811 14.90117 13.83042 13.64174
#Proportion of Variance  0.02335  0.0186  0.01461  0.01041  0.00897  0.00873
#Cumulative Proportion   0.68285  0.7015  0.71606  0.72648  0.73545  0.74418
#                           PC13    PC14     PC15     PC16     PC17    PC18
#Standard deviation     12.87364 11.6844 10.98917 10.69003 10.08273 9.95892
#Proportion of Variance  0.00777  0.0064  0.00566  0.00536  0.00477 0.00465
#Cumulative Proportion   0.75195  0.7583  0.76402  0.76938  0.77415 0.77880
#                          PC19    PC20    PC21    PC22    PC23    PC24    PC25
#Standard deviation     9.51313 9.25275 9.10213 8.97129 8.62083 8.48430 8.28904
#Proportion of Variance 0.00424 0.00402 0.00389 0.00377 0.00349 0.00338 0.00322
#Cumulative Proportion  0.78304 0.78706 0.79094 0.79472 0.79820 0.80158 0.80480

pca_summary <- summary(pca_res)$importance
write.csv(pca_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_all_samples.csv", row.names = TRUE)
write.table(pca_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_all_samples.txt", sep = "\t", row.names = TRUE)

# Calculate proportion of variance explained by each principal component
pve <- pca_res$sdev^2 / sum(pca_res$sdev^2)

# Create a scree plot
pve_df <- data.frame(
  PC_num = seq_along(pve),
  PC = paste0("PC", seq_along(pve)),
  VarianceExplained = pve
)

# Filter for the first 20 PCs
pve_df_first20 <- pve_df[pve_df$PC_num <= 20, ]

# Scree plot for the first 20 PCs
plot <- ggplot(pve_df_first20, aes(x = PC_num, y = VarianceExplained)) +
  geom_line() +
  geom_point() +
  labs(title = "Scree Plot (First 20 PCs)", 
       x = "Principal Component", 
       y = "Proportion of Variance Explained") +
  theme_minimal()

# Save the plot
ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/brain_PCA_scree_plot5.png",
       plot = plot, dpi = 300)

# Assume `merged_metadata` contains all your covariates
# Use first 8 PCs that explains 70% of variation
pcs_df <- as.data.frame(pca_res$x[, 1:8])
colnames(pcs_df) <- paste0("PC", 1:8)

# Combine PCs with metadata
pc_cov_df <- cbind(brain_metadata, pcs_df)
pc_cov_df <- as.data.frame(pc_cov_df)

pc_cov_df[, c("ISM_SEMA4", "RSM_SEMA4", "LIMS_SEMA4", "BARCODE_ISMMS", "IID_ISMMS","fastqDir",
          "s4newbatch","rapidBatch")] <- NULL

pc_cov_df[, c("FASTQC_Adapter_Content","FASTQC_Overrepresented_sequences",
"FASTQC_Per_base_sequence_content","FASTQC_Per_sequence_GC_content","FASTQC_Per_tile_sequence_quality",
"FASTQC_Sequence_Duplication_Levels")] <- NULL
# Run canonical correlation
cc_res <- canCorPairs(pc_cov_df)

print(cc_res)
plot(cc_res)

library(ggplot2)
##Iterative Covar Search
ggplot(pc_cov_df, aes(x = PC1, y = PC2, color = mymet_sex)) +
  geom_point(size = 2, alpha = 0.7) +
  theme_minimal() +
  labs(title = "PC1 vs PC2 colored by Sex")

# Update your model formula
form_updated <- ~ LIV_PM_Status + mymet_rin + new_covariate + (1 | IID_ISMMS) + (1 | mymet_sex)


In [ ]:
## playing with CPM filtering, leveraging the summary(), which gives column-wise summary
row_summary_list <- apply(cpm_blood, 1, summary) #for apply(), 1 = row and 2 = column
row_summary_df <- as.data.frame(t(row_summary_list))

head(row_summary_df, 10)

summary(row_summary_df$Min.) #summary of min is 
#    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
#    0.00     0.00     0.00     3.19     0.00 39257.60 
#75% of genes have minimum CPM = 0 across all samples
#Median = 0 so more than half the genes are not expressed at all in many samples
#So filtering genes that are "on" in only 1 or 2 samples makes sense

summary(row_summary_df$Mean)
#     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
#     0.00      0.04      0.15     16.97      1.51 147118.62 
#Median CPM across all genes is 0.15 (very low)


summary(row_summary_df$Median)
#     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
#     0.00      0.00      0.05     15.65      1.16 140084.17

In [ ]:
## grid search for BRAIN CPM THRESHOLD
library(edgeR)

# Assume cpm_blood is a matrix of CPM values (genes x samples)
dge_brain <- DGEList(counts = brain_ge)

# Convert to CPM
cpm_brain <- cpm(dge_brain)

n_samples <- ncol(cpm_brain)

# Expanded grid of thresholds (generous to moderate)
cpm_cutoffs <- c(0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0)
sample_percents <- c(0.05, 0.1, 0.15 ,0.2, 0.3, 0.5)

# Grid of CPM and sample proportion combinations
threshold_results <- expand.grid(CPM = cpm_cutoffs, SamplePercent = sample_percents)
threshold_results$GenesKept <- NA

# Loop through each combination
for (i in seq_len(nrow(threshold_results))) {
  cpm_cutoff <- threshold_results$CPM[i]
  sample_cutoff <- ceiling(threshold_results$SamplePercent[i] * n_samples)
  
  keep_genes <- rowSums(cpm_brain >= cpm_cutoff) >= sample_cutoff
  threshold_results$GenesKept[i] <- sum(keep_genes)
}

# Convert SamplePercent to % for clearer labeling
threshold_results$SamplePercentLabel <- scales::percent(threshold_results$SamplePercent)

threshold_results[1:12,1:4]
    CPM SamplePercent GenesKept SamplePercentLabel
1  0.10          0.05     35798                 5%
2  0.25          0.05     30090                 5%
3  0.50          0.05     26018                 5%
4  0.75          0.05     23637                 5%
5  1.00          0.05     22144                 5%
6  1.50          0.05     20084                 5%
7  2.00          0.05     18807                 5%
8  0.10          0.10     33899                10%
9  0.25          0.10     28764                10%
10 0.50          0.10     24859                10% ## use the same threshold as blood
11 0.75          0.10     22757                10%
12 1.00          0.10     21321                10%
13 1.50          0.10     19365                10%
14 2.00          0.10     18177                10%
15 0.10          0.15     32711                15%
16 0.25          0.15     27874                15%
17 0.50          0.15     24145                15%
18 0.75          0.15     22119                15%
19 1.00          0.15     20716                15%
20 1.50          0.15     18889                15%
21 2.00          0.15     17726                15%
22 0.10          0.20     31898                20%
23 0.25          0.20     27253                20%
24 0.50          0.20     23632                20%
25 0.75          0.20     21650                20%
26 1.00          0.20     20330                20%
27 1.50          0.20     18540                20%
28 2.00          0.20     17417                20%
29 0.10          0.30     30540                30%
30 0.25          0.30     26197                30%
31 0.50          0.30     22852                30%
32 0.75          0.30     20922                30%
33 1.00          0.30     19600                30%
34 1.50          0.30     17960                30%
35 2.00          0.30     16858                30%
36 0.10          0.50     28593                50%
37 0.25          0.50     24555                50%
38 0.50          0.50     21476                50%
39 0.75          0.50     19656                50%
40 1.00          0.50     18518                50%
41 1.50          0.50     16979                50%
42 2.00          0.50     15996                50%

# Plot results
a <- ggplot(threshold_results, aes(x = SamplePercent, y = GenesKept, color = factor(CPM))) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_x_continuous(labels = scales::percent) +
  scale_y_continuous(
    limits = c(10000, 55000),
    breaks = seq(10000, 55000, by = 5000)
  ) +
  labs(
    title = "Effect of CPM and Sample % Thresholds on Gene Retention",
    x = "Minimum % of Samples with CPM ≥ Cutoff",
    y = "Number of Genes Retained",
    color = "CPM Threshold"
  ) +
  theme_minimal(base_size = 14)

ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/brain_cpm_grid_search.png",
       plot = a, width = 10)

blood_keep <- rownames(cpm_blood)[rowSums(cpm_blood >= 0.5) >= ceiling(0.1 * ncol(cpm_blood))]
brain_keep <- rownames(cpm_brain)[rowSums(cpm_brain >= 0.5) >= ceiling(0.1 * ncol(cpm_brain))]

length(intersect(blood_keep, brain_keep))  # 21481 overlapped genes between blood and brain


In [ ]:
## NEW QC 5/9/2025 
####### BRAIN
# Convert raw counts to DGEList
dge_brain <- DGEList(counts = brain_ge)
# Convert to CPM
cpm_brain <- cpm(dge_brain)
# Filtering: Keep genes with CPM >= 1 in at least 10% of samples -- another decision point 
min_samples <- ceiling(0.1 * ncol(brain_ge))  # 10% of samples
keep_genes <- rowSums(cpm_brain >= 1.00) >= min_samples
dge_brain <- dge_brain[keep_genes, , keep.lib.sizes = FALSE]  # Apply filtering
vobj_brain <- voom(dge_brain, plot = FALSE) #this is normal 

# Apply TMM normalization for composition bias
dge_filtered_brain <- calcNormFactors(dge_brain, method = "TMM")

v_brain <- voomWithDreamWeights(dge_filtered_brain, formula = ~ 1, data = brain_metadata)

dim(v_brain$E) # 21356   233 | Keep genes with CPM >= 1 in at least 10% of samples 

#21321   287 | Keep genes with CPM >= 1 in at least 10% of samples 
#dim(v_brain$E) #24859   287 | Keep genes with CPM >= 0.5 in at least 10% of samples 

In [ ]:
##VARIANCE EXPLAINED BY EACH PARAMETER
##calculate variance explained for each variable in the metadata and determine the order in which this is plotted for PC1:PC2
#leveraging residual var to calculate var_explained by each variable in metadata (all 170)
## do it for each and every variables in metadata (all 170), use fixed effect to get an upper bound
fit <- lm(t(v$E) ~ 1 + mymet_age, blood_metadata) #change mymet_age to sth else
fraction_var_explained <- 1 - sum(colVars(fit$residual)) / sum(rowVars(v$E))

##running it for all variables in metadata
var_explained_list <- list()
skipped_vars <- list()  # To store skipped variable names and reasons

for (var_name in colnames(blood_metadata)) {
  # Extract the variable
  this_var <- blood_metadata[[var_name]]
  
  # Check if variable has only one level
  if (length(unique(this_var)) < 2) {
    skipped_vars[[var_name]] <- "Only one unique value"
    next
  }

  # Build formula
  formula_str <- paste0("t(v$E) ~ 1 + ", var_name)
  
  # Try fitting the model
  tryCatch({
    fit <- lm(as.formula(formula_str), data = blood_metadata)
    residual_var <- sum(colVars(fit$residuals))
    total_var <- sum(rowVars(v$E))
    frac_var_expl <- 1 - residual_var / total_var
    var_explained_list[[var_name]] <- frac_var_expl
  }, error = function(e) {
    skipped_vars[[var_name]] <- e$message
  })
}

# Create result dataframes
var_explained_df <- data.frame(
  Variable = names(var_explained_list),
  FractionExplained = unlist(var_explained_list)
)
var_explained_df <- var_explained_df[order(-var_explained_df$FractionExplained), ]
dim(var_explained_df) #161 x 2

# Convert skipped variable log to dataframe
skipped_df <- data.frame(
  Variable = names(skipped_vars),
  Reason = unlist(skipped_vars),
  row.names = NULL
)

dim(skipped_df) #9 x s2

# Optionally write skipped variables to file
# write.csv(skipped_df, "skipped_variables_log.csv", row.names = FALSE)

write.csv(var_explained_df,file= "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_variance_explained_results.csv", row.names = FALSE)
write.csv(skipped_df,file= "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_skipped_variables_log.csv", row.names = FALSE)

In [ ]:
# Perform PCA directly on normalized expression matrix - BLOOD
pca <- prcomp(t(v$E), center = TRUE, scale. = TRUE)

# View variance explained
summary(pca)
Importance of components:
                           PC1     PC2      PC3      PC4     PC5      PC6
Standard deviation     87.6509 47.3947 45.24788 32.50646 29.0587 24.78590
Proportion of Variance  0.3603  0.1053  0.09603  0.04956  0.0396  0.02881
Cumulative Proportion   0.3603  0.4657  0.56171  0.61127  0.6509  0.67969
                            PC7     PC8     PC9     PC10     PC11     PC12
Standard deviation     21.87412 18.9800 16.1963 13.67336 13.40437 12.96266
Proportion of Variance  0.02244  0.0169  0.0123  0.00877  0.00843  0.00788
Cumulative Proportion   0.70213  0.7190  0.7313  0.74010  0.74853  0.75641
                           PC13     PC14     PC15    PC16   PC17   PC18    PC19
Standard deviation     12.70339 11.20011 10.26162 9.93874 9.6829 9.5728 9.05608
Proportion of Variance  0.00757  0.00588  0.00494 0.00463 0.0044 0.0043 0.00385
Cumulative Proportion   0.76398  0.76986  0.77480 0.77944 0.7838 0.7881 0.79198
                         PC20    PC21    PC22    PC23    PC24    PC25    PC26
Standard deviation     8.8844 8.68013 8.54248 8.33063 8.15928 8.01271 7.73498
Proportion of Variance 0.0037 0.00353 0.00342 0.00325 0.00312 0.00301 0.00281
Cumulative Proportion  0.7957 0.79921 0.80264 0.80589 0.80901 0.81202 0.81483
                          PC27    PC28    PC29    PC30    PC31    PC32    PC33
Standard deviation     7.65527 7.60624 7.34043 7.28938 7.25140 7.16627 6.97011
Proportion of Variance 0.00275 0.00271 0.00253 0.00249 0.00247 0.00241 0.00228
Cumulative Proportion  0.81758 0.82029 0.82282 0.82531 0.82778 0.83019 0.83247
                          PC34    PC35    PC36   PC37    PC38    PC39    PC40
Standard deviation     6.90193 6.78131 6.62698 6.5272 6.50554 6.43177 6.34564
Proportion of Variance 0.00223 0.00216 0.00206 0.0020 0.00198 0.00194 0.00189
Cumulative Proportion  0.83470 0.83686 0.83892 0.8409 0.84290 0.84484 0.84673

##from Beckmann_lab_code.ipynb
resCor=canCorAllAgainstAll_Original(brain_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)

ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
brain_metadata2=brain_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
head(resCor,100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,2:5]))
# head(resCor[ordered_resCor,],100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,3:5]))
# head(resCor[ordered_resCor,],100)
library(ggrastr)

info_all2 <- brain_metadata2
dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
summ=summary(pca)
SampleByVariable = t(v$E)
clonename<-rownames(SampleByVariable)

In [ ]:
## grid search for BLOOD CPM THRESHOLD
library(edgeR)

dge_blood <- DGEList(counts = blood_ge) #creates 2 lists, dge_blood$counts &  dge_blood$samples 
#this propoerly tracks library sizes, required for downstream calcNormFactors, and keep counts and sample info tgt

# Convert to CPM
cpm_blood <- cpm(dge_blood) #log = TRUE

# Assume cpm_blood is a matrix of CPM values (genes x samples)
n_samples <- ncol(cpm_blood)

# Expanded grid of thresholds (generous to moderate)
cpm_cutoffs <- c(0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0)
sample_percents <- c(0.05, 0.1, 0.15 ,0.2, 0.3, 0.5)

# Grid of CPM and sample proportion combinations
threshold_results <- expand.grid(CPM = cpm_cutoffs, SamplePercent = sample_percents)
threshold_results$GenesKept <- NA

# Loop through each combination
for (i in seq_len(nrow(threshold_results))) {
  cpm_cutoff <- threshold_results$CPM[i]
  sample_cutoff <- ceiling(threshold_results$SamplePercent[i] * n_samples)
  
  keep_genes <- rowSums(cpm_blood >= cpm_cutoff) >= sample_cutoff
  threshold_results$GenesKept[i] <- sum(keep_genes)
}

# Convert SamplePercent to % for clearer labeling
threshold_results$SamplePercentLabel <- scales::percent(threshold_results$SamplePercent)

threshold_results 
    CPM SamplePercent GenesKept SamplePercentLabel
1  0.10          0.05     51934                 5%
2  0.25          0.05     40624                 5%
3  0.50          0.05     31435                 5%
4  0.75          0.05     26874                 5%
5  1.00          0.05     24061                 5%
6  1.50          0.05     20606                 5%
7  2.00          0.05     18474                 5%
8  0.10          0.10     46187                10%
9  0.25          0.10     34649                10%
10 0.50          0.10     27095                10% ## good start maybe (27095/58920 = 46% of all genes)
11 0.75          0.10     23473                10%
12 1.00          0.10     21163                10%
13 1.50          0.10     18322                10%
14 2.00          0.10     16597                10%
15 0.10          0.15     41624                15%
16 0.25          0.15     31110                15%
17 0.50          0.15     24699                15%
18 0.75          0.15     21457                15%
19 1.00          0.15     19431                15%
20 1.50          0.15     17015                15%
21 2.00          0.15     15621                15%
22 0.10          0.20     38029                20%
23 0.25          0.20     28656                20%
24 0.50          0.20     22934                20%
25 0.75          0.20     20084                20%
26 1.00          0.20     18311                20%
27 1.50          0.20     16225                20%
28 2.00          0.20     14946                20%
29 0.10          0.30     32692                30%
30 0.25          0.30     25179                30%
31 0.50          0.30     20526                30%
32 0.75          0.30     18261                30%
33 1.00          0.30     16872                30%
34 1.50          0.30     15196                30%
35 2.00          0.30     14085                30%
36 0.10          0.50     25391                50%
37 0.25          0.50     20816                50%
38 0.50          0.50     17813                50%
39 0.75          0.50     16268                50%
40 1.00          0.50     15219                50%
41 1.50          0.50     13923                50%
42 2.00          0.50     13002                50%

# Plot results
a <- ggplot(threshold_results, aes(x = SamplePercent, y = GenesKept, color = factor(CPM))) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_x_continuous(labels = scales::percent) +
  scale_y_continuous(
    limits = c(10000, 55000),
    breaks = seq(10000, 55000, by = 5000)
  ) +
  labs(
    title = "Effect of CPM and Sample % Thresholds on Gene Retention",
    x = "Minimum % of Samples with CPM ≥ Cutoff",
    y = "Number of Genes Retained",
    color = "CPM Threshold"
  ) +
  theme_minimal(base_size = 14)

ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/blood_cpm_grid_search.png",
       plot = a, width = 10)

#Need sufficient gene overlap between blood and brain, not necessarily on statistical testing, so being generous is reasonable
#Keep biologically meaningful genes, especially housekeeping and shared systemic signals that can travel or be co-regulated across tissues.

## BASED ON VOOM, GENES THAT ARE LOWLY EXPRESSED HAVE HIGHER VARIANCE 

In [ ]:
##old abd
form1 <- ~ (1|IID_ISMMS)
sub<-blood_only_metadata[,c("mymet_age_blood","mymet_sex_blood","RNASeqMetrics_MEDIAN_3PRIME_BIAS_blood", "mymet_rin_blood","STAR_Insertion_average_length_blood","Lymphocyte_total_wilk_blood")]

form7 <- ~ mymet_rin_blood + STAR_Insertion_average_length_blood
fit_blood <- dream(v_blood$E, form7, blood_only_metadata)
resid_expr_blood_form7 <- residuals(fit_blood)
cor_matrix7 <- abs(cor(t(resid_expr_blood_form7), t(v_brain$E), method = "spearman"))
cor_matrix7_mean <- mean(as.matrix(cor_matrix7)); cor_matrix7_mean #
cor_matrix7_97.5 <- quantile(as.matrix(cor_matrix7), probs = 0.975); cor_matrix7_97.5 #

#cor_matrix7_median <- median(as.matrix(cor_matrix7)); cor_matrix7_median  #
#cor_matrix7_80 <- quantile(as.matrix(cor_matrix7), probs = 0.8); cor_matrix7_80 #
#cor_matrix7_90 <- quantile(as.matrix(cor_matrix7), probs = 0.9); cor_matrix7_90 #
#cor_matrix7_92.5 <- quantile(as.matrix(cor_matrix7), probs = 0.925); cor_matrix7_92.5 #
#cor_matrix7_95 <- quantile(as.matrix(cor_matrix7), probs = 0.95); cor_matrix7_95 #

form7 <- ~ mymet_age_blood + mymet_sex_blood + RNASeqMetrics_MEDIAN_3PRIME_BIAS_blood + mymet_rin_blood + STAR_Insertion_average_length_blood + Lymphocyte_total_wilk_blood
#Spearman mean = 0.05634181 | 97.5% = 0.1547861

form7 <- ~ mymet_age_blood + mymet_sex_blood + mymet_race_blood + RNASeqMetrics_MEDIAN_3PRIME_BIAS_blood + mymet_rin_blood + STAR_Insertion_average_length_blood + Lymphocyte_total_wilk_blood
#Spearman mean = 0.05640757 | 97.5% = 0.1544933 | Pearson mean = 0.05692875 | 97.5% = 0.1549941


write.csv(resid_expr_blood_form1, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form1.csv", row.names = TRUE)
test<-read.csv("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form1.csv")

write.table(resid_expr_blood_form1, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form1.txt", sep = "\t", quote = FALSE, col.names = NA)
test2<-fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form1.txt",data.table=FALSE)

form2 <- ~ (1|IID_ISMMS) + mymet_rin_blood
fit_blood <- dream(v_blood$E, form2, blood_metadata)
resid_expr_blood_form2 <- residuals(fit_blood)

write.csv(resid_expr_blood_form2, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form2.csv", row.names = TRUE)
write.table(resid_expr_blood_form2, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form2.txt", sep = "\t", quote = FALSE, col.names = NA)

form3 <- ~ (1|IID_ISMMS) + mymet_rin_blood + STAR_Insertion_average_length_blood
fit_blood <- dream(v_blood$E, form3, blood_metadata)
resid_expr_blood_form3 <- residuals(fit_blood)

write.csv(resid_expr_blood_form3, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form3.csv", row.names = TRUE)
write.table(resid_expr_blood_form3, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_form3.txt", sep = "\t", quote = FALSE, col.names = NA)

In [ ]:
## NEW QC 5/5/2025
####### BLOOD
# Convert raw counts to DGEList
dge_blood <- DGEList(counts = blood_ge) #creates 2 lists, dge_blood$counts &  dge_blood$samples 
#this propoerly tracks library sizes, required for downstream calcNormFactors, and keep counts and sample info tgt

# Convert to CPM
cpm_blood <- cpm(dge_blood) #log = TRUE

# Filtering: Keep genes with CPM >= 1 in at least 10% of samples ## need to play around with this threshold
#min_samples <- ceiling(0.1 * ncol(blood_ge)) # 10% of samples | good is CPM >= 1 in 50% of sample but this depends on the research question 
min_samples <- ceiling(0.1 * ncol(blood_ge)) # 10% of samples | good is CPM >= 1 in 50% of sample but this depends on the research question 
keep_genes <- rowSums(cpm_blood >= 1.00) >= min_samples ## change this from 1 to 0.5
dge_blood <- dge_blood[keep_genes, , keep.lib.sizes = FALSE]  # Apply filtering

vobj_blood <- voom(dge_blood, plot = FALSE) # this is normal

# OPTION 2 (PREFERRED): Apply voomWithDreamWeights with your full model
dge_filtered_blood <- calcNormFactors(dge_blood, method = "TMM")

#log_cpm_mat <- cpm(dge_filtered, log=TRUE, prior.count = 0.5) 
v_blood <- voomWithDreamWeights(dge_filtered_blood, formula = ~1, data = blood_metadata)

dim(v_blood$E) #21046   233
#21163   243 | Keep genes with CPM >= 1 in at least 10% of samples 
#dim(v_blood$E) #27095   243 | Keep genes with CPM >= 0.5 in at least 10% of samples #lowly expressed genes might be helpful
#v$targets  v$E        v$weights 
save.image("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood-brain_sample_baseline_with_voom_20250522.RData")

load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood-brain_sample_baseline_with_voom_20250522.RData")


In [ ]:
## streamlined PROCESS
## REMOVED 8 SAMPLES FOR NOW
blood_samples_to_be_removed <- c("LBPSEMA4BLOOD795", "LBPSEMA4BLOOD142", "LBPSEMA4BLOOD049", "LBPSEMA4BLOOD755",
                                  "LBPSEMA4BLOOD335","LBPSEMA4BLOOD431","LBPSEMA4BLOOD174","LBPSEMA4BLOOD738")

brain_samples_to_be_removed <- c("LBPSEMA4BRAIN364", "LBPSEMA4BRAIN318", "LBPSEMA4BRAIN564", "LBPSEMA4BRAIN170",
                                  "LBPSEMA4BRAIN703", "LBPSEMA4BRAIN341","LBPSEMA4BRAIN391","LBPSEMA4BRAIN017")


v_blood$E <- v_blood$E[, !colnames(v_blood$E) %in% blood_samples_to_be_removed]
dim(v_blood$E) #21046   225
blood_metadata <- blood_metadata[!rownames(blood_metadata) %in% blood_samples_to_be_removed, ]
dim(blood_metadata) #225 339
blood_only_metadata <- blood_metadata[, grepl("_blood$", names(blood_metadata))]
blood_only_metadata <- blood_only_metadata[!rownames(blood_only_metadata) %in% blood_samples_to_be_removed, ]
dim(blood_only_metadata) #225 169

v_brain$E <- v_brain$E[, !colnames(v_brain$E) %in% brain_samples_to_be_removed]
dim(v_brain$E) #21356   225
brain_metadata <- brain_metadata[!rownames(brain_metadata) %in% brain_samples_to_be_removed, ]
dim(brain_metadata) #225 339
brain_only_metadata <- brain_metadata[, grepl("_brain$", names(brain_metadata))]
brain_only_metadata <- brain_only_metadata[!rownames(brain_only_metadata) %in% brain_samples_to_be_removed, ]
dim(brain_only_metadata) #225 169

## correlation
cor_matrix <- cor(t(v_blood$E), t(v_brain$E), method = "spearman")
dim(cor_matrix) 
write.csv(cor_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.csv")
write.table(cor_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.txt", sep = "\t", quote = FALSE, row.names = TRUE, col.names = NA)

## pca
pca_brain <- prcomp(t(v_brain$E), center = TRUE, scale. = TRUE)
pca_blood <- prcomp(t(v_blood$E), center = TRUE, scale. = TRUE)

pca_brain_summary <- summary(pca_brain)$importance
pca_blood_summary <- summary(pca_blood)$importance

write.csv(pca_brain_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_paired_samples_20250522.csv", row.names = TRUE)
write.table(pca_brain_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_paired_samples_20250522.txt", sep = "\t", row.names = TRUE)

write.csv(pca_blood_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_paired_samples_20250522.csv", row.names = TRUE)
write.table(pca_blood_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_paired_samples_20250522.txt", sep = "\t", row.names = TRUE)

## plotting
# run canCorAllAgainstAll_Original() and plot_pca_by_metadata()

### BLOOD
blood_only_metadata <- blood_metadata[, grepl("_blood$", names(blood_metadata))]
pca = pca_blood
resCor=canCorAllAgainstAll_Original(blood_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
blood_only_metadata = blood_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
#head(resCor,100)
info_all2 <- blood_only_metadata
summ=summary(pca)
SampleByVariable = t(v_blood$E)
clonename<-rownames(SampleByVariable)

plot_pca_by_metadata(pca_blood, summary(pca_blood), info_all2, clonename,
                     file_prefix = "blood", form_label = "form1")

### BRAIN
brain_only_metadata <- brain_metadata[, grepl("_brain$", names(brain_metadata))]
pca = pca_brain
resCor=canCorAllAgainstAll_Original(brain_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
brain_only_metadata = brain_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
info_all2 <- brain_only_metadata
summ=summary(pca)
SampleByVariable = t(v_brain$E)
clonename<-rownames(SampleByVariable)

plot_pca_by_metadata(pca_brain, summary(pca_brain), info_all2, clonename,
                     file_prefix = "brain", form_label = "form0")

In [ ]:
## 5-27-2025
## in my thesis proposal, I performed correlations between ONLY SHARED genes, so the 17k genes that are in both blood and brain 
## for this analysis, it's more comprehensive and im looking at correlations of all genes, including those in 1 tissue

base_cor <- fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.txt", data.table=FALSE)
dim(base_cor) #21046 21357
base_cor[1:5,1:5]
row.names(base_cor) <- base_cor$V1
base_cor$V1 <- NULL

brain_by_blood <- t(base_cor)
dim(brain_by_blood) #21356 21046
sub<-brain_by_blood[1:5,1:5]

## plot distribution of the correlation -- i did this for my thesis?
cor_values <- as.vector(brain_by_blood)

png("/hpc/users/hoangd02/www/plots/lbp/blood_brain_cor_distribution_baseline.png") #looks pretty normal
hist(cor_values, breaks = 100, main = "Distribution of Blood-Brain Gene Expression Correlations", 
     xlab = "Spearman Correlation", col = "gray")
dev.off()

# Convert to data frame
cor_df <- data.frame(correlation = cor_values)
p <- ggplot(cor_df, aes(y = correlation)) +
  geom_boxplot(fill = "gray") +
  coord_flip() +
  scale_y_continuous(breaks = seq(-1, 1, 0.2)) +
  labs(title = "Distribution of Blood-Brain Gene Expression Correlations",
       x = "", y = "Spearman Correlation") +
  theme_minimal()

ggsave("/hpc/users/hoangd02/www/plots/lbp/blood_brain_cor_distribution_baseline_boxplot.png",
       plot = p, width = 10, height = 4)

#summary(sub) #summary() does column-wise summary
row_summary <- t(apply(brain_by_blood, 1, function(x) {
  quant <- quantile(x, probs = c(0, 0.25, 0.5, 0.75, 1), na.rm = TRUE)
  mn <- mean(x, na.rm = TRUE)
  c(Min = quant[1], `1st Qu.` = quant[2], Median = quant[3], Mean = mn,
    `3rd Qu.` = quant[4], Max = quant[5])
}))
colnames(row_summary) <- c("Min.", "1st Qu.", "Median", "Mean", "3rd Qu.", "Max.")

row_summary[1:5,1:5]

row_summary_highest_mean <- row_summary[order(abs(row_summary[, "Mean"]), decreasing = TRUE), ]
row_summary_highest_mean[1:5,1:6]
#max mean: 0.03652670

row_summary_highest_median <- row_summary[order(abs(row_summary[, "Median"]), decreasing = TRUE), ]
row_summary_highest_median[1:5,1:6]
#max median: 0.05166437 

row_summary_highest_max <- row_summary[order(abs(row_summary[, "Max."]), decreasing = TRUE), ]
row_summary_highest_max[1:5,1:6]
#max max: 0.8225240

load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_blood_qc2.RData")
resid_expr_blood[,"LBPSEMA4BLOOD795"] #outliers do not exist
# this version already removed outliers
dim(resid_expr_blood) #21046   225

dim(resid_expr_brain) #21356   225

cor_matrix_resid <- cor(t(resid_expr_blood), t(resid_expr_brain), method = "spearman")
dim(cor_matrix_resid) #21046 21356
write.csv(cor_matrix_resid, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix_resid_groupID.csv")
write.table(cor_matrix_resid, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix_resid_groupID.txt", sep = "\t", quote = FALSE, row.names = TRUE, col.names = NA)

##make sure distribution of cor is still normal (aka doesn't change)
cor_values <- as.vector(cor_matrix_resid)
png("/hpc/users/hoangd02/www/plots/lbp/blood_brain_cor_distribution_resid_groupID.png") #looks pretty normal
hist(cor_values, breaks = 100, main = "Distribution of Blood-Brain Gene Expression Correlations", 
     xlab = "Spearman Correlation", col = "gray")
dev.off()

row_summary <- t(apply(cor_matrix_resid, 1, function(x) {
  quant <- quantile(x, probs = c(0, 0.25, 0.5, 0.75, 1), na.rm = TRUE)
  mn <- mean(x, na.rm = TRUE)
  c(Min = quant[1], `1st Qu.` = quant[2], Median = quant[3], Mean = mn,
    `3rd Qu.` = quant[4], Max = quant[5])
}))
colnames(row_summary) <- c("Min.", "1st Qu.", "Median", "Mean", "3rd Qu.", "Max.")
row_summary_highest_mean <- row_summary[order(abs(row_summary[, "Mean"]), decreasing = TRUE), ]
row_summary_highest_mean[1:5,1:6]
#max mean: 0.03652670 --> 0.01071001

row_summary_highest_median <- row_summary[order(abs(row_summary[, "Median"]), decreasing = TRUE), ]
row_summary_highest_median[1:5,1:6]
#max median: 0.05166437 --> -0.01446692

row_summary_highest_max <- row_summary[order(abs(row_summary[, "Max."]), decreasing = TRUE), ]
row_summary_highest_max[1:5,1:6]
#max max: 0.8225240 --> 0.4751054



In [ ]:
## ran this after removing outliers
# residualize, pca, cor, repeat
### BLOOD
form1 <- ~ (1|IID_ISMMS)
form2 <- ~ (1|IID_ISMMS) + mymet_rin_blood
fit_blood <- dream(v_blood$E, form1, blood_metadata)
resid_expr_blood <- residuals(fit_blood)

#resid_expr_blood_resid_ID_rin <- resid_expr_blood
#write.table(resid_expr_blood_resid_ID_rin, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_resid_ID_rin_20250522.txt", sep = "\t", row.names = TRUE)

#resid_expr_blood_resid_ID <- resid_expr_blood
#write.table(resid_expr_blood_resid_ID, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/resid_expr_blood_resid_ID_20250522.txt", sep = "\t", row.names = TRUE)

## do correlations btw these and v_brain$E using form0
#resid_expr_blood_resid_ID
#resid_expr_blood_resid_ID_rin 
pca_resid_blood <- prcomp(t(resid_expr_blood), center = TRUE, scale. = TRUE)
#Total:798 s
#Warning message:
#In .standard_transform(ret) :
#  No testable fixed effects were included in the model.
#  Running topTable() will fail.
pca = pca_resid_blood
resCor=canCorAllAgainstAll_Original(blood_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
blood_only_metadata = blood_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
#head(resCor,100)
info_all2 <- blood_only_metadata
summ=summary(pca)
SampleByVariable = t(resid_expr_blood) ###change this 
clonename<-rownames(SampleByVariable)

plot_pca_by_metadata(pca, summ, info_all2, clonename,
                     file_prefix = "blood", form_label = "form1")

### BRAIN -- 
## 2 models, one with form0 with just intercept and the other with 
form2 <- ~ (1|IID_ISMMS) + mymet_rin_brain
fit_brain <- dream(v_brain$E, form1, brain_metadata)
resid_expr_brain <- residuals(fit_brain)
pca_resid_brain <- prcomp(t(resid_expr_brain), center = TRUE, scale. = TRUE)
pca = pca_resid_brain 
resCor=canCorAllAgainstAll_Original(brain_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
brain_only_metadata = brain_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
info_all2 <- brain_only_metadata
summ=summary(pca)
SampleByVariable = t(resid_expr_brain) ###change this 
clonename<-rownames(SampleByVariable)

#plot_pca_by_metadata(pca, summ, info_all2, clonename,
#                     file_prefix = "brain", form_label = "form1")

plot_pca_by_metadata(pca, summ, info_all2, clonename,
                                       file_prefix = "brain", form_label = "form1")

save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_blood_qc.RData")
save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_brain_qc.RData")

##the blood_metadata here still have outliers 
load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_brain_qc.RData")
blood_metadata["LBPSEMA4BLOOD795", ]

##later version at night: 2025-05-22
save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_blood_qc2.RData")
save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_brain_qc2.RData")

load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/play_resid_brain_qc2.RData")

LBPSEMA4BLOOD795
blood_metadata["LBPSEMA4BLOOD795", ]

https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_blood_form0_2025-05-22.pdf
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_blood_form1_2025-05-22residualizedID.pdf
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_brain_form1_2025-05-22residualizedID.pdf

##Questions - 
1. What's wrong w braingenie QC? PEER? it's not data-driven/reiterative? 
2. What's the difference with regressing variables vs accounting for it as covariates in a model?
3. Regress out cell type composition vs regress out ODC_brain etc. in brain?
4. Anina's brain model - how did she account for cell type composition if not cibersortx?
5. Sex - pretty low correlation but known effect, but add or not? Prob add
6. "repeating this process until no more strong confounders remained" what's strong here?
7. Dream is pretty slow ~15 mins each time
8. Cibersortx: LIV reference vs PM reference, seems inconclusive @Lora's dissertation

In [ ]:
## Outlier detection 
find_pca_outliers <- function(pca, clonename, pc_x = 1, pc_y = 2, level = pnorm(3) - pnorm(-3)) {
  # Extract selected PCs
  pc_df <- data.frame(
    X = pca$x[, pc_x],
    Y = pca$x[, pc_y],
    ID = clonename
  )
  
  # Compute Mahalanobis distance
  center <- colMeans(pc_df[, c("X", "Y")])
  cov_matrix <- cov(pc_df[, c("X", "Y")])
  pc_df$mahal_dist <- mahalanobis(pc_df[, c("X", "Y")], center, cov_matrix)
  
  # Convert confidence level (e.g. 99.73%) to chi-squared cutoff with 2 df
  chisq_cutoff <- qchisq(level, df = 2)
  
  # Flag outliers
  pc_df$outlier <- pc_df$mahal_dist > chisq_cutoff
  
  return(pc_df)
}

outlier_df <- find_pca_outliers(pca, clonename, pc_x = 4, pc_y = 5)
outlier_ids <- outlier_df$ID[outlier_df$outlier]
outlier_ids 

## BLOOD OUTLIERS
# PC1 vs PC2
"LBPSEMA4BLOOD795"

# PC2 vs PC3
[1] "LBPSEMA4BLOOD142" "LBPSEMA4BLOOD795" "LBPSEMA4BLOOD755"

# PC3 vs PC4
[1] "LBPSEMA4BLOOD142" "LBPSEMA4BLOOD795" "LBPSEMA4BLOOD755"

# PC4 vs PC5
[1] "LBPSEMA4BLOOD795" "LBPSEMA4BLOOD049" "LBPSEMA4BLOOD755"

## BRAIN OUTLIERS
# PC1 vs PC2
"LBPSEMA4BRAIN364"
# PC2 vs PC3
"LBPSEMA4BRAIN364"
# PC3 vs PC4
[1] "LBPSEMA4BRAIN318" "LBPSEMA4BRAIN564"
# PC4 vs PC5
[1] "LBPSEMA4BRAIN318" "LBPSEMA4BRAIN170" "LBPSEMA4BRAIN564"

## USE THE DICTIONARY
blood_brain_id_dict <- blood_metadata[,c("IID_ISMMS","SAMPLE_ISMMS_blood","SAMPLE_ISMMS_brain")]
head(blood_brain_id_dict)
blood_outliers <- c("LBPSEMA4BLOOD795", "LBPSEMA4BLOOD142", "LBPSEMA4BLOOD049", "LBPSEMA4BLOOD755")
brain_outliers <- c("LBPSEMA4BRAIN364", "LBPSEMA4BRAIN318", "LBPSEMA4BRAIN564", "LBPSEMA4BRAIN170")

blood_brain_id_dict[blood_brain_id_dict$SAMPLE_ISMMS_blood %in% blood_outliers, ]
#                 IID_ISMMS SAMPLE_ISMMS_blood SAMPLE_ISMMS_brain
#LBPSEMA4BLOOD142   PT-0028   LBPSEMA4BLOOD142   LBPSEMA4BRAIN703
#LBPSEMA4BLOOD795   PT-0113   LBPSEMA4BLOOD795   LBPSEMA4BRAIN341
#LBPSEMA4BLOOD049   PT-0126   LBPSEMA4BLOOD049   LBPSEMA4BRAIN391
#LBPSEMA4BLOOD755   PT-0162   LBPSEMA4BLOOD755   LBPSEMA4BRAIN017

blood_brain_id_dict[blood_brain_id_dict$SAMPLE_ISMMS_brain %in% brain_outliers, ]
#                 IID_ISMMS SAMPLE_ISMMS_blood SAMPLE_ISMMS_brain
#LBPSEMA4BLOOD335   PT-0179   LBPSEMA4BLOOD335   LBPSEMA4BRAIN364
#LBPSEMA4BLOOD431   PT-0024   LBPSEMA4BLOOD431   LBPSEMA4BRAIN318
#LBPSEMA4BLOOD174   PT-0029   LBPSEMA4BLOOD174   LBPSEMA4BRAIN170
#LBPSEMA4BLOOD738   PT-0123   LBPSEMA4BLOOD738   LBPSEMA4BRAIN564

## REMOVED 8 SAMPLES FOR NOW
blood_samples_to_be_removed <- c("LBPSEMA4BLOOD795", "LBPSEMA4BLOOD142", "LBPSEMA4BLOOD049", "LBPSEMA4BLOOD755",
                                  "LBPSEMA4BLOOD335","LBPSEMA4BLOOD431","LBPSEMA4BLOOD174","LBPSEMA4BLOOD738")

brain_samples_to_be_removed <- c("LBPSEMA4BRAIN364", "LBPSEMA4BRAIN318", "LBPSEMA4BRAIN564", "LBPSEMA4BRAIN170",
                                  "LBPSEMA4BRAIN703", "LBPSEMA4BRAIN341","LBPSEMA4BRAIN391","LBPSEMA4BRAIN017")


v_blood$E <- v_blood$E[, !colnames(v_blood$E) %in% blood_samples_to_be_removed]
dim(v_blood$E) #21046   225
blood_metadata <- blood_metadata[!rownames(blood_metadata) %in% blood_samples_to_be_removed, ]
dim(blood_metadata) #225 339
blood_only_metadata <- blood_only_metadata[!rownames(blood_only_metadata) %in% blood_samples_to_be_removed, ]
dim(blood_only_metadata) #225 169

v_brain$E <- v_brain$E[, !colnames(v_brain$E) %in% brain_samples_to_be_removed]
dim(v_brain$E) #21356   225
brain_metadata <- brain_metadata[!rownames(brain_metadata) %in% brain_samples_to_be_removed, ]
dim(brain_metadata) #225 339
brain_only_metadata <- brain_only_metadata[!rownames(brain_only_metadata) %in% brain_samples_to_be_removed, ]
dim(brain_only_metadata) #225 169


In [ ]:
#Run canCorAllAgainstAll_Original  - Jolie's updated code 
canCorAllAgainstAll_Original <- function(X, Y = X, minimum_intersect = 0) {
  library(stringr)

  # Create model matrices for each variable
  X_formulas <- lapply(str_c("~", colnames(X)), as.formula)
  X_varList <- lapply(X_formulas, function(xf) model.matrix.lm(xf, X, na.action = "na.pass")[, -1, drop = FALSE])

  Y_formulas <- lapply(str_c("~", colnames(Y)), as.formula)
  Y_varList <- lapply(Y_formulas, function(yf) model.matrix.lm(yf, Y, na.action = "na.pass")[, -1, drop = FALSE])

  # Initialize result matrix
  XY_cc <- matrix(nrow = ncol(X), ncol = ncol(Y), data = NA,
                  dimnames = list(colnames(X), colnames(Y)))

  for (ix in seq_along(X_varList)) {
    keep1 <- apply(X_varList[[ix]], 1, function(x) !any(is.na(x)))
    for (iy in seq_along(Y_varList)) {
      keep2 <- apply(Y_varList[[iy]], 1, function(x) !any(is.na(x)))
      keep <- keep1 & keep2

      x_mat <- X_varList[[ix]][keep, , drop = FALSE]
      y_mat <- Y_varList[[iy]][keep, , drop = FALSE]

      # Extra safe check: enough data, non-zero variance, and full rank
      if (sum(keep) > minimum_intersect &&
          ncol(x_mat) > 0 && ncol(y_mat) > 0 &&
          any(apply(x_mat, 2, var, na.rm = TRUE) > 0) &&
          any(apply(y_mat, 2, var, na.rm = TRUE) > 0)) {

        # Try cancor, safely
        try_result <- tryCatch({
          fit <- cancor(x_mat, y_mat)
          sqrt(mean(fit$cor^2))
        }, error = function(e) NA)

        XY_cc[ix, iy] <- try_result
      } else {
        XY_cc[ix, iy] <- NA
      }
    }
  }

  return(XY_cc)
}

In [ ]:
## OFFICIAL: FUNCTION -ALFY THE PLOTTING - Jolie's Updated Code
plot_pca_by_metadata <- function(pca, summ, info_all2, clonename, 
                                 file_prefix, form_label,
                                 type = "norm") {
  require(ggplot2)
  require(ggrepel)
  require(gridExtra)
  require(patchwork)
  require(sp)
  require(ggrastr)
  
  # True confidence levels for ±1, 2, 3 SD
  level1 <- pnorm(1) - pnorm(-1)  # ~68.27%
  level2 <- pnorm(2) - pnorm(-2)  # ~95.45%
  level3 <- pnorm(3) - pnorm(-3)  # ~99.73%

  total <- ncol(info_all2)
  count <- 1
  dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
  out_path <- paste0("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",
                     file_prefix, "_", form_label, "_", dateFreeze, "residualizedID_rin.pdf")
  
  pdf(out_path, width = 10, height = 8)
  
  for (col in colnames(info_all2)) {
    cat("on column", count, "/", total, col, "\n")
    color_data <- info_all2[[col]]
    
    plot_pair <- function(x, y, pc_x, pc_y, col_name, is_discrete) {
      base <- ggplot(data.frame(pca$x), aes_string(x = pc_x, y = pc_y,
                                                   colour = if (is_discrete) paste0("factor(info_all2[['", col_name, "']])")
                                                            else paste0("info_all2[['", col_name, "']]"),
                                                   label = "clonename")) +
        theme_bw() +
        rasterize(geom_point(size = 0.8)) +
        labs(title = paste0("PC", x, "-PC", y),
             x = paste0("PC", x, ": ", round(summ$importance[2, x] * 100, digits = 2), "%"),
             y = paste0("PC", y, ": ", round(summ$importance[2, y] * 100, digits = 2), "%")) +
        ggtitle(col_name) +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level3, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level2, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level1, linetype = "dotdash", colour = "darkgrey")
      
      if (is_discrete) {
        if (length(unique(color_data)) < 7) {
          base <- base +
            scale_colour_discrete(guide = "legend", name = substr(col_name, 1, 6)) +
            theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
            guides(colour = guide_legend(override.aes = list(size = 2)))
        } else {
          base <- base + scale_colour_discrete(guide = "none")
        }
      } else {
        base <- base + scale_colour_gradientn(guide = "legend",
                                               name = substr(col_name, 1, 6),
                                               colours = rainbow(2)) +
          theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
          guides(colour = guide_legend(override.aes = list(size = 2)))
      }
      return(base)
    }
    
    is_discrete <- class(color_data) != "numeric"
    
    a <- plot_pair(1, 2, "pca$x[,1]", "pca$x[,2]", col, is_discrete)
    b <- plot_pair(2, 3, "pca$x[,2]", "pca$x[,3]", col, is_discrete)
    c <- plot_pair(3, 4, "pca$x[,3]", "pca$x[,4]", col, is_discrete)
    d <- plot_pair(4, 5, "pca$x[,4]", "pca$x[,5]", col, is_discrete)

    print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count <- count + 1
  }
  
  dev.off()
  cat("PDF saved to:", out_path, "\n")
}

In [ ]:
### PCA PLOT FUNCTION WITH OUTLIERS DETECTION
plot_pca_by_metadata <- function(pca, summ, info_all2, clonename, 
                                 file_prefix, form_label,
                                 type = "norm") {
  require(ggplot2)
  require(ggrepel)
  require(gridExtra)
  require(patchwork)
  require(sp)
  require(ggrastr)
  
  # True confidence levels for ±1, 2, 3 SD
  level1 <- pnorm(1) - pnorm(-1)  # ~68.27%
  level2 <- pnorm(2) - pnorm(-2)  # ~95.45%
  level3 <- pnorm(3) - pnorm(-3)  # ~99.73%

  total <- ncol(info_all2)
  count <- 1
  dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
  out_path <- paste0("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",
                     file_prefix, "_", form_label, "_", dateFreeze, "secondtry.pdf")
  
  pdf(out_path, width = 10, height = 8)
  outlier_list <- list()
  
  for (col in colnames(info_all2)) {
    cat("on column", count, "/", total, col, "\n")
    color_data <- info_all2[[col]]
    
    plot_pair <- function(x, y, pc_x, pc_y, col_name, is_discrete) {
      base <- ggplot(data.frame(pca$x), aes_string(x = pc_x, y = pc_y,
                                                   colour = if (is_discrete) paste0("factor(info_all2[['", col_name, "']])")
                                                            else paste0("info_all2[['", col_name, "']]"),
                                                   label = "clonename")) +
        theme_bw() +
        rasterize(geom_point(size = 0.8)) +
        labs(title = paste0("PC", x, "-PC", y),
             x = paste0("PC", x, ": ", round(summ$importance[2, x] * 100, digits = 2), "%"),
             y = paste0("PC", y, ": ", round(summ$importance[2, y] * 100, digits = 2), "%")) +
        ggtitle(col_name) +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level3, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level2, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level1, linetype = "dotdash", colour = "darkgrey")
      
      if (is_discrete) {
        if (length(unique(color_data)) < 7) {
          base <- base +
            scale_colour_discrete(guide = "legend", name = substr(col_name, 1, 6)) +
            theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
            guides(colour = guide_legend(override.aes = list(size = 2)))
        } else {
          base <- base + scale_colour_discrete(guide = "none")
        }
      } else {
        base <- base + scale_colour_gradientn(guide = "legend",
                                               name = substr(col_name, 1, 6),
                                               colours = rainbow(2)) +
          theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
          guides(colour = guide_legend(override.aes = list(size = 2)))
      }
      return(base)
    }
    
    is_discrete <- class(color_data) != "numeric"
    
    a <- plot_pair(1, 2, "pca$x[,1]", "pca$x[,2]", col, is_discrete)
    b <- plot_pair(2, 3, "pca$x[,2]", "pca$x[,3]", col, is_discrete)
    c <- plot_pair(3, 4, "pca$x[,3]", "pca$x[,4]", col, is_discrete)
    d <- plot_pair(4, 5, "pca$x[,4]", "pca$x[,5]", col, is_discrete)

    # Detect outliers from PC1-PC2 plot using only level3 ellipse
    build <- ggplot_build(a)$data
    points <- build[[1]]
    ellipses <- build[[3]]
    
    # Safely get only the ellipse with highest level (most outer)
    if ("level" %in% colnames(ellipses)) {
      ell <- subset(ellipses, abs(level - level3) < 1e-6)
    } else {
      warning("No level column found in ellipse data — skipping outlier detection for column: ", col)
      ell <- NULL
    }
    
    if (!is.null(ell) && nrow(ell) > 2) {
      dat <- data.frame(x = points$x, y = points$y, label = points$label,
                        in_ell = as.logical(point.in.polygon(points$x, points$y, ell$x, ell$y)))
      outliers <- points$label[!dat$in_ell]
      outlier_list[[col]] <- outliers
    } else {
      outlier_list[[col]] <- character(0)  # no outliers detected
    }

    print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count <- count + 1
  }
  
  dev.off()
  cat("PDF saved to:", out_path, "\n")
  return(outlier_list)
}


In [ ]:
##WORKED: PCA PLOT FUNCTION WITH OUTLIERS DETECTION, the one that detected 8 outliers in blood
plot_pca_by_metadata <- function(pca, summ, info_all2, clonename, 
                                 file_prefix, form_label,
                                 type = "norm") {
  require(ggplot2)
  require(ggrepel)
  require(gridExtra)
  require(patchwork)
  require(sp)
  require(ggrastr)
  
  # True confidence levels for ±1, 2, 3 SD
  level1 <- pnorm(1) - pnorm(-1)  # ~68.27%
  level2 <- pnorm(2) - pnorm(-2)  # ~95.45%
  level3 <- pnorm(3) - pnorm(-3)  # ~99.73%

  total <- ncol(info_all2)
  count <- 1
  dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
  out_path <- paste0("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",
                     file_prefix, "_", form_label, "_", dateFreeze, "secondtry.pdf")
  
  pdf(out_path, width = 10, height = 8)
  
  # Initialize list to store outliers
  outlier_list <- list()
  
  for (col in colnames(info_all2)) {
    cat("on column", count, "/", total, col, "\n")
    color_data <- info_all2[[col]]
    
    plot_pair <- function(x, y, pc_x, pc_y, col_name, is_discrete) {
      base <- ggplot(data.frame(pca$x), aes_string(x = pc_x, y = pc_y,
                                                   colour = if (is_discrete) paste0("factor(info_all2[['", col_name, "']])")
                                                            else paste0("info_all2[['", col_name, "']]"),
                                                   label = "clonename")) +
        theme_bw() +
        rasterize(geom_point(size = 0.8)) +
        labs(title = paste0("PC", x, "-PC", y),
             x = paste0("PC", x, ": ", round(summ$importance[2, x] * 100, digits = 2), "%"),
             y = paste0("PC", y, ": ", round(summ$importance[2, y] * 100, digits = 2), "%")) +
        ggtitle(col_name) +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level3, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level2, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level1, linetype = "dotdash", colour = "darkgrey")
      
      if (is_discrete) {
        if (length(unique(color_data)) < 7) {
          base <- base +
            scale_colour_discrete(guide = "legend", name = substr(col_name, 1, 6)) +
            theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
            guides(colour = guide_legend(override.aes = list(size = 2)))
        } else {
          base <- base + scale_colour_discrete(guide = "none")
        }
      } else {
        base <- base + scale_colour_gradientn(guide = "legend",
                                               name = substr(col_name, 1, 6),
                                               colours = rainbow(2)) +
          theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
          guides(colour = guide_legend(override.aes = list(size = 2)))
      }
      return(base)
    }
    
    is_discrete <- class(color_data) != "numeric"
    
    a <- plot_pair(1, 2, "pca$x[,1]", "pca$x[,2]", col, is_discrete)
    b <- plot_pair(2, 3, "pca$x[,2]", "pca$x[,3]", col, is_discrete)
    c <- plot_pair(3, 4, "pca$x[,3]", "pca$x[,4]", col, is_discrete)
    d <- plot_pair(4, 5, "pca$x[,4]", "pca$x[,5]", col, is_discrete)

    # Detect outliers from PC1-PC2 plot
    build <- ggplot_build(a)$data
    points <- build[[1]]
    ell <- build[[3]]
    dat <- data.frame(x = points$x, y = points$y, label = points$label,
                      in_ell = as.logical(point.in.polygon(points$x, points$y, ell$x, ell$y)))
    outliers <- points$label[!dat$in_ell]
    outlier_list[[col]] <- outliers

    print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count <- count + 1
  }
  
  dev.off()
  cat("PDF saved to:", out_path, "\n")
  return(outlier_list)
}


In [ ]:
## BASELINE correlations between blood genes and brain genes - MAY 22 2025
dim(v_blood$E) # 21046   233
dim(v_brain$E) # 21356   233 

write.table(v_blood$E, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_expression_voom_paired_samples_20250522.txt", sep="\t", quote=FALSE, row.names=TRUE)
write.table(v_brain$E, "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_expression_voom_paired_samples_20250522.txt", sep="\t", quote=FALSE, row.names=TRUE)

# Transpose: now rows = samples, columns = genes
blood_t <- t(v_blood$E)  # dim: 233 x 21046
brain_t <- t(v_brain$E)  # dim: 233 x 21356

# Spearman correlation across samples (rows)
# Result: 21046 x 21356 matrix (blood genes x brain genes)
cor_matrix <- cor(blood_t, brain_t, method = "spearman")

#cor_matrix[i, j] gives the Spearman correlation between blood gene i and brain gene j across the 233 individuals.
dim(cor_matrix) # 21046 BLOOD genes x 21356 BRAIN genes

cor_matrix[1:5,1:5]

saveRDS(cor_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.rds")
write.csv(cor_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.csv")
write.table(cor_matrix, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_brain_spearman_cor_matrix.txt", sep = "\t", quote = FALSE, row.names = TRUE, col.names = NA)
#cor_matrix <- readRDS("blood_brain_spearman_cor_matrix.rds")

summary(cor_matrix)
cor_matrix_first_10genes <- cor_matrix[1:10,1:10]
summary(cor_matrix_first_10genes) #column level summary
apply(cor_matrix_first_10genes, 1, summary) #row level summary

In [ ]:
##BASELINE PCA 
level3=pnorm(3,mean=0,sd=1,lower.tail=T) - pnorm(3,lower.tail=F) #Probability that a Z-score is within ±3 standard deviations of the mean
level2=pnorm(2,mean=0,sd=1,lower.tail=T) - pnorm(2,lower.tail=F)
level1=pnorm(1,mean=0,sd=1,lower.tail=T) - pnorm(1,lower.tail=F)
type="norm"

options(width=150)
library(gridExtra)
library(patchwork)
library(sp)

# Perform PCA
pca_brain <- prcomp(t(v_brain$E), center = TRUE, scale. = TRUE)
pca_blood <- prcomp(t(v_blood$E), center = TRUE, scale. = TRUE)

# variance explained
pca_brain_summary <- summary(pca_brain)$importance
pca_brain_summary[1:3,1:11]
#                      V1      PC1      PC2      PC3      PC4      PC5      PC6
#1     Standard deviation 87.79333 50.89971 44.36194 33.04898 28.04109 23.93262
#2 Proportion of Variance  0.36091  0.12131  0.09215  0.05114  0.03682  0.02682
#3  Cumulative Proportion  0.36091  0.48223  0.57438  0.62552  0.66234  0.68916
#       PC7      PC8      PC9     PC10
#1 20.68322 18.44141 16.70067 14.36477
#2  0.02003  0.01592  0.01306  0.00966
#3  0.70919  0.72512  0.73818  0.74784
write.csv(pca_brain_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_paired_samples_20250522.csv", row.names = TRUE)
write.table(pca_brain_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_paired_samples_20250522.txt", sep = "\t", row.names = TRUE)

test <- fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/brain_pca_summary_paired_samples_20250522.txt",data.table=FALSE)
test[1:3,1:11]

pca_blood_summary <- summary(pca_blood)$importance
#                      V1      PC1      PC2      PC3      PC4      PC5      PC6
#1     Standard deviation 64.70919 55.50087 46.31027 38.55534 34.22353 30.17755
#2 Proportion of Variance  0.19896  0.14636  0.10190  0.07063  0.05565  0.04327
#3  Cumulative Proportion  0.19896  0.34532  0.44722  0.51786  0.57351  0.61678
#       PC7      PC8      PC9     PC10
#1 25.37435 20.91020 19.01961 17.43576
#2  0.03059  0.02078  0.01719  0.01444
#3  0.64737  0.66815  0.68533  0.69978
write.csv(pca_blood_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_paired_samples_20250522.csv", row.names = TRUE)
write.table(pca_blood_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_paired_samples_20250522.txt", sep = "\t", row.names = TRUE)
test <- fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_paired_samples_20250522.txt",data.table=FALSE)
test[1:3,1:11]

In [ ]:
## MANUALLY Generating the plots
# run this one at a time:
library(ggrastr)

### EITHER RUN FOR BLOOD OR BRAIN!

### BLOOD
dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
blood_only_metadata <- blood_metadata[, grepl("_blood$", names(blood_metadata))]
pca = pca_blood
resCor=canCorAllAgainstAll_Original(blood_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
blood_only_metadata = blood_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
#head(resCor,100)
info_all2 <- blood_only_metadata
summ=summary(pca)
SampleByVariable = t(v_blood$E)
clonename<-rownames(SampleByVariable)
pdf(paste("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",dateFreeze,"_secondtry.pdf",sep="")) #this is for blood

### BRAIN
dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
brain_only_metadata <- brain_metadata[, grepl("_brain$", names(brain_metadata))]
pca = pca_brain
resCor=canCorAllAgainstAll_Original(brain_only_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
brain_only_metadata = brain_only_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
info_all2 <- brain_only_metadata
summ=summary(pca)
SampleByVariable = t(v_brain$E)
clonename<-rownames(SampleByVariable)
pdf(paste("/hpc/users/hoangd02/www/plots/lbp/pca_plots_brain_",dateFreeze,"_secondtry.pdf",sep="")) #this is for brain

count=1
total=ncol(info_all2)
  for(col in colnames(info_all2)){
    cat("on column",count,"/",total,col,"\n")
  # #=======pca-1 vs pca-2=======
    if(class(info_all2[,col])!="numeric"){
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + #geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      if(length(unique(info_all2[,col]))<7){
        a = a + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        a = a + scale_colour_discrete(guide ="none")
      }
      build <- ggplot_build(a)$data
      points <- build[[1]]
      ell <- build[[3]]

      # Find which points are inside the ellipse, and add this to the data
      dat <- data.frame(points[1:2], 
                        in.ell = as.logical(point.in.polygon(points$x, points$y, ell$x, ell$y)))
      outliers_3SD_PC1_PC2=points$label[which(dat$in.ell==F)]
      # show(a)
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + 
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        b = b + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        b = b + scale_colour_discrete(guide ="none")
      }
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + 
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        c = c + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        c = c + scale_colour_discrete(guide ="none")
      }
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + #geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        d = d + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        d = d + scale_colour_discrete(guide ="none")
      }
    }else{
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      a = a +
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      b = b + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      c = c + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      d = d + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
    }
    # show(a)
    # show(b)
    # show(c)
    # show(d)
   #multiplot_same_legend(a,b,c,d,cols=2)
   (a + b) / (c + d) + plot_layout(guides = "collect")
   print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count=count+1
  }
dev.off()

save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_with_corr_and_pc.RData")

load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_with_corr_and_pc.RData")

https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_2025-05-08.pdf
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_2025-05-08.png
##
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_2025-05-22_secondtry.pdf
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_brain_2025-05-22_secondtry.pdf


In [ ]:
## SKIP - try on a subset of genes first, always do this!! 
#compute the Spearman correlation of expression patterns across samples, 
#for gene A in blood vs gene B in brain.
common_genes <- intersect(rownames(v_blood$E), rownames(v_brain$E))
length(common_genes) #17533 genes

percent_overlap <- (length(common_genes) / length(unique(c(rownames(v_blood$E), rownames(v_brain$E))))) * 100
#denominator = 24869 genes
length(unique(c(rownames(v_blood$E), rownames(v_brain$E)))) #24869 genes
# Genes that are in Either: 24869 genes
percent_overlap #70.50143

v_blood$E[1:5,1:5]
v_brain$E[1:5,1:5]

################# CORRELATIONS 
############BEFORE RESIDUALIZING FOR COVARIATES
# Subset both matrices to only include common genes
blood_expression_sub <- v_blood$E[common_genes, ] #17581   243
brain_expression_sub <- v_brain$E[common_genes, ] # 17581   287

## to do spearman correlations, the 2 matrices have to have the same dimensions
blood_ge <- raw_count[, colnames(raw_count) %in% blood_metadata$SAMPLE_ISMMS_blood]
dim(blood_ge)
#58929   233

brain_ge <- raw_count[, colnames(raw_count) %in% brain_metadata$SAMPLE_ISMMS_brain]
dim(brain_ge)

subset_genes <- shared_genes[1:100]
cor_matrix_test <- matrix(NA, nrow = 100, ncol = 100)
rownames(cor_matrix_test) <- subset_genes
colnames(cor_matrix_test) <- subset_genes

for (i in seq_along(subset_genes)) {
  gene_blood <- subset_genes[i]
  
  cor_matrix_test[i, ] <- sapply(subset_genes, function(gene_brain) {
    cor(v_blood$E[gene_blood, ], v_brain$E[gene_brain, ], method = "spearman")
  })
  
  if (i %% 10 == 0) {
    cat(sprintf("[%s] Processed %d of %d genes\n", 
                format(Sys.time(), "%H:%M:%S"), i, length(subset_genes)))
  }
}

##
library(matrixStats)

blood_gene_median <- rowMedians(v_blood$E)
brain_gene_median <- rowMedians(v_brain$E)

set.seed(2025)  # for reproducibility
blood_subset <- blood_gene_median[1:100]
brain_subset <- brain_gene_median[1:100]

cor_matrix_test <- matrix(NA, nrow = length(blood_subset), ncol = length(brain_subset))
rownames(cor_matrix_test) <- names(blood_subset)
colnames(cor_matrix_test) <- names(brain_subset)
cor_matrix_test[1:5,1:5]

for (i in seq_along(blood_subset)) {
  gene_blood <- as.numeric(blood_subset[i])
  
  cor_matrix_test[i, ] <- sapply(brain_subset, function(gene_brain) {
    cor(gene_blood, as.numeric(gene_brain), method = "spearman")
  })
  
  if (i %% 10 == 0) {
    cat(sprintf("[%s] Processed %d of %d genes\n", 
                format(Sys.time(), "%H:%M:%S"), i, length(blood_subset)))
  }
}


In [ ]:
## SKIPPP -- these are baseline measurements, have not add any covariates
dim(v_blood$E) #27095 genes x 243 samples
dim(v_brain$E) #24859 genes x 287 samples

#this is just the expression levels 
blood_summary <- apply(v_blood$E, 1, summary) #for apply(), 1 = row and 2 = column
blood_summary_df <- as.data.frame(t(blood_summary ))
head(blood_summary_df, 10)
#summary(blood_summary_df$Median) #summary of Median
#   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
#-4.9233 -1.7228  0.7644  0.9681  3.5504 17.0959

library(matrixStats)

blood_gene_median <- rowMedians(v_blood$E)
brain_gene_median <- rowMedians(v_brain$E)

# Optional: name them for matching
names(blood_gene_median) <- rownames(v_blood$E)
names(brain_gene_median) <- rownames(v_brain$E)

common_genes <- intersect(names(blood_gene_median), names(brain_gene_median))

same_gene_corr <- cor(blood_gene_median[common_genes],
                      brain_gene_median[common_genes],
                      method = "spearman")
#Calculating the full gene–gene Spearman correlation matrix (e.g., 27k × 24k matrix!)
cor_matrix <- outer(
  blood_gene_median, brain_gene_median,
  Vectorize(function(x, y) cor(x, y, method = "spearman"))
)
rownames(cor_matrix) <- names(blood_gene_median)
colnames(cor_matrix) <- names(brain_gene_median)

##modified version with checks
# Initialize result matrix
cor_matrix <- matrix(NA, nrow = length(blood_gene_median), ncol = length(brain_gene_median))
rownames(cor_matrix) <- names(blood_gene_median)
colnames(cor_matrix) <- names(brain_gene_median)

# Start timer
start_time <- Sys.time()
cat("Starting Spearman correlation computation at", format(start_time), "\n")

# Loop through blood genes
for (i in seq_along(blood_gene_median)) {
  gene_blood <- blood_gene_median[i]
  
  # Compute correlation with all brain genes
  cor_matrix[i, ] <- sapply(brain_gene_median, function(gene_brain) {
    cor(gene_blood, gene_brain, method = "spearman")
  })
  
  # Print progress every 100 genes
  if (i %% 100 == 0 || i == length(blood_gene_median)) {
    current_time <- Sys.time()
    cat(sprintf("[%s] Processed %d of %d blood genes\n",
                format(current_time, "%H:%M:%S"), i, length(blood_gene_median)))
  }
}

# End timer
end_time <- Sys.time()
cat("Finished at", format(end_time), " | Total time:", round(difftime(end_time, start_time, units = "mins"), 2), "minutes\n")

write.table(
  cor_matrix,
  file = "cor_matrix.txt.gz",
  sep = "\t",
  quote = FALSE,
  col.names = NA,
  row.names = TRUE
)

#MIGHT NEED 
##another version: If your goal is to optimize blood→brain prediction, 
#then use the average Spearman correlation between blood genes and 
#their best-correlated brain gene. 
# Create matrix: genes in rows, genes in columns (blood vs brain)
cor_mat <- outer(
  blood_gene_median,
  brain_gene_median,
  Vectorize(function(x, y) cor(x, y, method = "spearman"))
)

# For each blood gene, find its best-correlated brain gene
max_corr_per_blood_gene <- apply(cor_mat, 1, function(x) max(abs(x), na.rm = TRUE))

# Average of maximum absolute correlations
mean_best_corr <- mean(max_corr_per_blood_gene, na.rm = TRUE)

#other things to try: Cosine similarity or Canonical Correlation (CCA)
#- Canonical Correlation Analysis (CCA): Maximize shared variance between blood and brain gene spaces
#- Cosine similarity between ranked gene profiles
#- Regression performance (cross-validated R²) from blood to brain

## how many genes in the blood are more correlated in the brain 

## distribution of correlations 

## why do we regress confounders out and not just add them as covariates?


cor_matrix.txt.gz

library(data.table)
cor_matrix <- fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_exploration/cor_matrix.txt.gz", data.table = FALSE)

cor_matrix[1:5, 1:5]



















In [ ]:
## VARIANCE EXPLAINED BLOOD
##calculate variance explained for each variable in the metadata and determine the order in which this is plotted for PC1:PC2
#threshold --- ?

#leveraging residual var to calculate var_explained by each variable in metadata (all 170)
## do it for each and every variables in metadata (all 170), use fixed effect to get an upper bound
fit <- lm(t(v$E) ~ 1 + mymet_age, blood_metadata) 
fraction_var_explained <- 1 - sum(colVars(fit$residual)) / sum(rowVars(v$E))

##running it for all variables in metadata
var_explained_list <- list()
skipped_vars <- list()  # To store skipped variable names and reasons

for (var_name in colnames(blood_metadata)) {
  # Extract the variable
  this_var <- blood_metadata[[var_name]]
  
  # Check if variable has only one level
  if (length(unique(this_var)) < 2) {
    skipped_vars[[var_name]] <- "Only one unique value"
    next
  }

  # Build formula
  formula_str <- paste0("t(v$E) ~ 1 + ", var_name)
  
  # Try fitting the model
  tryCatch({
    fit <- lm(as.formula(formula_str), data = blood_metadata)
    residual_var <- sum(colVars(fit$residuals))
    total_var <- sum(rowVars(v$E))
    frac_var_expl <- 1 - residual_var / total_var
    var_explained_list[[var_name]] <- frac_var_expl
  }, error = function(e) {
    skipped_vars[[var_name]] <- e$message
  })
}

# Create result dataframes
var_explained_df <- data.frame(
  Variable = names(var_explained_list),
  FractionExplained = unlist(var_explained_list)
)
var_explained_df <- var_explained_df[order(-var_explained_df$FractionExplained), ]
dim(var_explained_df) #161 x 2

# Convert skipped variable log to dataframe
skipped_df <- data.frame(
  Variable = names(skipped_vars),
  Reason = unlist(skipped_vars),
  row.names = NULL
)

dim(skipped_df) #9 x s2

# Optionally write skipped variables to file
# write.csv(skipped_df, "skipped_variables_log.csv", row.names = FALSE)

write.csv(var_explained_df,file= "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_variance_explained_results.csv", row.names = FALSE)
write.csv(skipped_df,file= "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_skipped_variables_log.csv", row.names = FALSE)

In [ ]:
## PCA BLOOD
# Perform PCA directly on normalized expression matrix
pca_results <- prcomp(t(v$E), center = TRUE, scale. = TRUE)

# View variance explained
summary(pca_results)
#Importance of components:
#                           PC1     PC2      PC3      PC4     PC5      PC6
#Standard deviation     64.9526 56.9769 45.78370 38.22967 33.9311 29.88761
#Proportion of Variance  0.1993  0.1534  0.09905  0.06906  0.0544  0.04221
#Cumulative Proportion   0.1993  0.3528  0.45180  0.52086  0.5753  0.61747
#                           PC7      PC8      PC9    PC10     PC11     PC12
#Standard deviation     25.1987 20.80907 19.31579 17.4547 16.82334 14.85197
#Proportion of Variance  0.0300  0.02046  0.01763  0.0144  0.01337  0.01042
#Cumulative Proportion   0.6475  0.66793  0.68556  0.7000  0.71333  0.72375
#                          PC13     PC14     PC15     PC16    PC17     PC18
#Standard deviation     13.4911 12.86578 12.15012 11.48534 10.9815 10.56168
#Proportion of Variance  0.0086  0.00782  0.00698  0.00623  0.0057  0.00527
#Cumulative Proportion   0.7324  0.74018  0.74715  0.75339  0.7591  0.76435
#                           PC19     PC20    PC21    PC22    PC23    PC24
#Standard deviation     10.34875 10.04605 9.80615 9.43519 9.27946 9.11003
#Proportion of Variance  0.00506  0.00477 0.00454 0.00421 0.00407 0.00392
#Cumulative Proportion   0.76942  0.77418 0.77873 0.78293 0.78700 0.79092
#                          PC25    PC26    PC27    PC28   PC29    PC30    PC31
#Standard deviation     8.75566 8.65861 8.53087 8.33230 8.2234 8.07722 7.90927
#Proportion of Variance 0.00362 0.00354 0.00344 0.00328 0.0032 0.00308 0.00296
#Cumulative Proportion  0.79455 0.79809 0.80153 0.80481 0.8080 0.81109 0.81404

pca<- pca_results
# Create a dataframe for the first few PCs
pca_data <- as.data.frame(pca_results$x[, 1:10])  # First 10 PCs
pca_data$SAMPLE_ISMMS <- rownames(pca_data)
#var_explained_for_each_PC <- pca_results$sdev^2 / sum(pca_results$sdev^2) * 100 # Computes the percent variance explained by each PC
# Variance explained for each PC = (stdev)^2 / total variance x 100
# Var_explained = numeric venctor showing how much of the total variation is captured by each PC

# Merge with metadata to explore relationships
pca_with_meta <- merge(pca_data, blood_metadata, by = "SAMPLE_ISMMS")

# Plot PC1 vs PC2 colored by potential confounding variables
p1 <- ggplot(pca_with_meta, aes(x = PC1, y = PC2, color = IID_ISMMS)) +
  geom_point(size = 3, alpha = 0.7) +
  theme_bw() +
  labs(title = "PCA plot colored by IID_ISMMS",
       x = paste0("PC1 (19.93%)"),
       y = paste0("PC2 (15.34%)"))

ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/PC1_PC2_by_IID_ISMMS.png",
       plot = p1, width=20, height=20)

#color by every variable in var_explained_df
table(blood_metadata$IID_ISMMS) #some individual have 2 samples some have 1 sample

p2 <- ggplot(pca_with_meta, aes(x = PC1, y = PC2, color = STAR_Insertion_average_length)) +
  geom_point(size = 3, alpha = 0.7) +
  theme_bw() +
  labs(title = "PCA plot colored by STAR_Insertion_average_length",
       x = paste0("PC1 (19.93%)"),
       y = paste0("PC2 (15.34%)"))

ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/PC1_PC2_by_STAR_Insertion_average_length.png",
       plot = p2, width=20, height=20)

## Streamline this -- change PCs in 3 places
library(ggplot2)
library(rlang)  # for .data

# Loop through each column in pca_with_meta
for (var in colnames(pca_with_meta)) {
  # Skip PC columns
  if (grepl("^PC\\d+$", var)) next
  
  # Skip variables with only one unique value
  if (length(unique(pca_with_meta[[var]])) <= 1) next

  # Plot and save
  p <- ggplot(pca_with_meta, aes(x = PC1, y = PC5, color = .data[[var]])) +
    geom_point(size = 6, alpha = 0.7) +
    theme_bw() +
    labs(title = paste("PCA plot colored by", var),
         x = "PC1 (19.93%)",
         #y = "PC2 (15.34%)")+ 
         #y = "PC3 (9.91%)") + 
         #y = "PC4 (6.9%)") + 
        y = "PC5 (5.4%)") + 
    theme(
        plot.title = element_text(size = 30, face = "bold", hjust = 0.5),
        axis.title = element_text(size = 24),
        axis.text = element_text(size = 20),
        legend.title = element_text(size = 22),
        legend.text = element_text(size = 18)
    )

  filename <- paste0("/hpc/users/hoangd02/www/plots/lbp/PC1_PC5_by_", var, ".png")
  ggsave(filename = filename, plot = p, width = 25, height = 25, dpi = 300)
}

In [ ]:
# Calculate correlations between PCs and metadata variables
# Select numeric variables from metadata
numeric_vars <- blood_metadata %>% 
  select_if(is.numeric)

# Calculate correlations with first 10 PCs
pc_correlations <- data.frame(variable = colnames(numeric_vars))

for(i in 1:10) {
  pc_name <- paste0("PC", i)
  cors <- apply(numeric_vars, 2, function(x) cor(x, pca_results$x[,i], use="pairwise.complete.obs"))
  pc_correlations[, pc_name] <- cors
}

# Display top correlations for each PC
for(i in 1:5) {  # First 5 PCs
  pc_name <- paste0("PC", i)
  cat("Top correlations with", pc_name, ":\n")
  sorted_cors <- sort(abs(pc_correlations[, pc_name]), decreasing = TRUE)
  print(head(sorted_cors, 5))
  cat("\n")
}

In [ ]:
evaluate_pc_correlations <- function(expression_matrix, metadata) {
  # Ensure the CCA package is available
  if (!requireNamespace("CCA", quietly = TRUE)) {
    stop("The CCA package is required for this function. Please install it using: install.packages('CCA')")
  }
  
  # Run PCA
  pca <- prcomp(t(expression_matrix), center = TRUE, scale. = TRUE) # Input = samples as rows
  pcs <- as.data.frame(pca$x[, 1:10]) # Extract the first 10 PCs
  var_explained <- pca$sdev^2 / sum(pca$sdev^2) * 100 # Percent variance explained by each PC
  
  # Set up logging for skipped variables and test selection
  log_messages <- character()
  
  # Prepare results dataframe
  results <- data.frame(
    variable = character(),
    pc = character(),
    correlation = numeric(),
    p_value = numeric(),
    var_explained = numeric(),
    test_type = character(),
    stringsAsFactors = FALSE
  )
  
  # Loop through metadata variables
  for (var_name in colnames(metadata)) {
    var_data <- metadata[[var_name]]
    
    # Check if there are enough data points
    if (sum(!is.na(var_data)) < 3) {
      log_messages <- c(log_messages, paste0("SKIPPED: Variable '", var_name, "' - Fewer than 3 non-NA values"))
      next
    }
    
    # Determine variable type and apply appropriate correlation test
    if (is.numeric(var_data)) {
      # For numeric variables: use Spearman correlation as before
      test_type <- "spearman"
      log_messages <- c(log_messages, paste0("SELECTED: Variable '", var_name, "' - Numeric variable using Spearman correlation"))
      
    } else if (is.factor(var_data) || is.character(var_data)) {
      # For categorical variables
      n_levels <- length(unique(na.omit(var_data)))
      
      if (n_levels <= 1) {
        log_messages <- c(log_messages, paste0("SKIPPED: Variable '", var_name, "' - Only one level found"))
        next
      }
      
      if (n_levels == 2) {
        # Binary categorical: convert to 0/1 and use Spearman
        var_data <- as.numeric(as.factor(var_data)) - 1
        test_type <- "binary_spearman"
        log_messages <- c(log_messages, paste0("SELECTED: Variable '", var_name, "' - Binary categorical converted to 0/1 for Spearman correlation"))
        
      } else {
        # Try to convert to ordinal if possible (all levels can be interpreted as numbers)
        levels_as_char <- as.character(unique(na.omit(var_data)))
        if (all(!is.na(suppressWarnings(as.numeric(levels_as_char))))) {
          # Ordinal categorical: convert to numeric and use Spearman
          var_data <- as.numeric(as.character(var_data))
          test_type <- "ordinal_spearman"
          log_messages <- c(log_messages, paste0("SELECTED: Variable '", var_name, "' - Ordinal categorical (", n_levels, " levels) converted to numeric for Spearman correlation"))
        } else {
          # Truly categorical with >2 levels: use CCA
          test_type <- "cca"
          log_messages <- c(log_messages, paste0("SELECTED: Variable '", var_name, "' - Multi-level categorical (", n_levels, " levels) using Canonical Correlation Analysis"))
        }
      }
    } else {
      # Skip other variable types
      log_messages <- c(log_messages, paste0("SKIPPED: Variable '", var_name, "' - Unsupported variable type"))
      next
    }
    
    # Calculate correlation/association with each PC
    for (i in 1:10) {
      pc_name <- paste0("PC", i)
      pc_data <- pcs[[pc_name]]
      
      # Get complete cases for both var_data and pc_data
      complete_idx <- which(!is.na(var_data) & !is.na(pc_data) & is.finite(var_data) & is.finite(pc_data))
      if (length(complete_idx) < 3) {
        log_messages <- c(log_messages, paste0("SKIPPED: Variable '", var_name, "' with PC", i, " - Fewer than 3 complete cases after removing NA/Inf values"))
        next
      }
      
      # Perform appropriate test based on variable type
      if (test_type %in% c("spearman", "binary_spearman", "ordinal_spearman")) {
        # Use Spearman correlation for numeric, binary, and ordinal
        cor_test <- tryCatch({
          cor.test(var_data[complete_idx], pc_data[complete_idx], method = "spearman")
        }, error = function(e) {
          log_messages <<- c(log_messages, paste0("ERROR: Variable '", var_name, "' with PC", i, " - Spearman correlation failed: ", e$message))
          NULL
        })
        
        if (!is.null(cor_test)) {
          results <- rbind(results, data.frame(
            variable = var_name,
            pc = pc_name,
            correlation = cor_test$estimate,
            p_value = cor_test$p.value,
            var_explained = var_explained[i],
            test_type = test_type
          ))
        }
        
      } else if (test_type == "cca") {
        # For truly categorical variables with >2 levels: use CCA
        cca_test <- tryCatch({
          # Create dummy variables for the categorical variable
          cat_data <- var_data[complete_idx]
          
          # Create dummy variables matrix
          dummy_matrix <- model.matrix(~ cat_data - 1)
          
          # Run CCA between dummy variables and PC
          # Prepare data for CCA
          X <- as.matrix(dummy_matrix)
          Y <- matrix(pc_data[complete_idx], ncol = 1)
          
          # Calculate canonical correlation using the CCA package
          cc_result <- CCA::cancor(X, Y)
          
          # Extract canonical correlation coefficient
          # Use the first (and only) canonical correlation
          cc_cor <- cc_result$cor[1]
          
          # Calculate p-value using Wilks' Lambda approximation
          n <- length(complete_idx)
          p <- ncol(X)
          q <- 1  # just one PC
          
          # Wilks' Lambda statistic
          wilks_lambda <- prod(1 - cc_result$cor^2)
          
          # Calculate chi-square statistic
          chi_square <- -(n - 1 - (p + q + 1)/2) * log(wilks_lambda)
          df <- p * q
          p_val <- 1 - pchisq(chi_square, df)
          
          list(estimate = cc_cor, p.value = p_val)
        }, error = function(e) {
          # If CCA fails, log the error and return NULL
          log_messages <<- c(log_messages, paste0("ERROR: Variable '", var_name, "' with PC", i, " - CCA failed: ", e$message))
          NULL
        })
        
        if (!is.null(cca_test)) {
          results <- rbind(results, data.frame(
            variable = var_name,
            pc = pc_name,
            correlation = cca_test$estimate, # This is canonical correlation for CCA
            p_value = cca_test$p.value,
            var_explained = var_explained[i],
            test_type = test_type
          ))
        }
      }
    }
  }
  
  # Adjust p-values and sort results
  if (nrow(results) > 0) {
    results$p_adj <- p.adjust(results$p_value, method = "BH")
    results <- results[order(abs(results$correlation), decreasing = TRUE), ]
  } else {
    log_messages <- c(log_messages, "WARNING: No valid correlations were found for any variable")
  }
  
  # Add log messages to the results
  return(list(
    pca = pca, 
    correlations = results,
    log = log_messages
  ))
}

In [ ]:
results <- evaluate_pc_correlations(v$E, blood_metadata)
results$pca           results$correlations  results$log           

head(results$correlations)

# Save the complete results object
saveRDS(results, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_pc_correlation_results.rds")

# Later, to load it:
test <- readRDS("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_pc_correlation_results.rds")

In [ ]:
# Function to evaluate PC correlations with metadata variables
evaluate_pc_correlations <- function(expression_matrix, metadata) {
  # Run PCA
  pca <- prcomp(t(expression_matrix), center = TRUE, scale. = TRUE) # Input = samples as rows
  pcs <- as.data.frame(pca$x[, 1:10]) # Extract the first 10 PCs
  var_explained <- pca$sdev^2 / sum(pca$sdev^2) * 100 # Computes the percent variance explained by each PC
  # Variance explained for each PC = (stdev)^2 / total variance x 100
  # Var_explained = numeric venctor showing how much of the total variation is captured by each PC
  
  # Prepare results dataframe
  results <- data.frame(variable = character(), pc = character(), 
                       correlation = numeric(), p_value = numeric(),
                       var_explained = numeric(), stringsAsFactors = FALSE)
  
  # Loop through metadata variables
  for (var_name in colnames(metadata)) {
    if (!is.numeric(metadata[[var_name]])) next # Only performing correlation with numeric variables
    # for categorical, convert 0 and 1 or 0, 1, 2 etc. Spearman
    # for truly categorical, do canonical correlation, take absolute 

    # Calculate correlation with each PC
    for (i in 1:10) {
      pc_name <- paste0("PC", i) # Constructs PC names - PC1, PC2, etc.
      complete_idx <- which(!is.na(metadata[[var_name]]) & is.finite(metadata[[var_name]])) #Remove samples with missing and non-fine values
      
      if (length(complete_idx) < 3) next # Skips correlation if fewer than 3 data points remain 
      
      cor_test <- tryCatch({
        cor.test(metadata[[var_name]][complete_idx], pcs[[pc_name]][complete_idx], 
                method = "spearman") # Perform Spearman correlation
      }, error = function(e) NULL)
      
      if (!is.null(cor_test)) {
        results <- rbind(results, data.frame( # Store results
          variable = var_name, pc = pc_name, correlation = cor_test$estimate,
          p_value = cor_test$p.value, var_explained = var_explained[i]
        ))
      }
    }
  }
  
  results$p_adj <- p.adjust(results$p_value, method = "BH")
  results <- results[order(abs(results$correlation), decreasing = TRUE), ] # Sort results 
  
  return(list(pca = pca, correlations = results))
}

# Run the analysis
initial_analysis <- evaluate_pc_correlations(v$E, blood_metadata)
top_correlations <- initial_analysis$correlations %>%
  arrange(p_adj, desc(abs(correlation))) %>%
  head(10)
print(top_correlations)


## Iterative identification of confounding variables
# Iteratively performing PCA + correlation between metadata variables and expression PCs
# Identifying the strongest significant covariate and Regress it out
# Repeating the process on the residual expression matrix

identify_covariates_iteratively <- function(expr_matrix, metadata, max_iter = 5, p_threshold = 0.01) {
  current_matrix <- expr_matrix
  selected_vars <- c()
  
  for (iter in 1:max_iter) { 
    # Run correlation analysis
    analysis <- evaluate_pc_correlations(current_matrix, metadata) # Run the function above
    
    # Find top significant variable
    sig_vars <- analysis$correlations %>%
      filter(p_adj < p_threshold) %>% # Filter p_adj for FDR < 0.01
      arrange(p_adj, desc(abs(correlation)))
    
    if (nrow(sig_vars) == 0 || all(sig_vars$variable %in% selected_vars)) break # Stop if there is no more significant var or all significant ones have been selected
    
    # Select top variable not already included
    new_vars <- sig_vars$variable[!sig_vars$variable %in% selected_vars]

    if (length(new_vars) == 0) break
    
    top_var <- new_vars[1] # Identify the top new variable that's not in the existed covar list
    cat("Iteration", iter, "- Selected variable:", top_var, "\n")
    selected_vars <- c(selected_vars, top_var) # Add the new covar to existing covar list
    
    # Create model formula with all selected variables
    if (length(selected_vars) > 0) {
      formula_str <- paste("~", paste(selected_vars, collapse = " + ")) # Build formula for limma linear model 
      design <- model.matrix(as.formula(formula_str), data = metadata)
      fit <- lmFit(current_matrix, design)
      current_matrix <- residuals(fit, current_matrix) # Computes the residuals (expression variation after adjusting for covariates)
    }
  }
  return(selected_vars)
}

# Run the iterative process
identified_covariates <- identify_covariates_iteratively(v$E, blood_metadata)
#Iteration 1 - Selected variable: Lymphocyte_total_wilk 
#Iteration 2 - Selected variable: RNASeqMetrics_MEDIAN_3PRIME_BIAS 
#Iteration 3 - Selected variable: STAR_Number_of_splices_Annotated__sjdb_ 
#Iteration 4 - Selected variable: STAR_Number_of_reads_unmapped_other 
#Iteration 5 - Selected variable: RNASeqMetrics_PCT_INTRONIC_BASES 

# Function to get detailed PC correlations for a specific variable
get_variable_pc_details <- function(analysis_results, variable_name) {
  # Filter correlations for the specific variable
  var_cors <- analysis_results$correlations %>%
    filter(variable == variable_name) %>%
    arrange(p_adj)
  
  # Return the details
  return(var_cors)
}

# Create a function to print detailed correlation information
print_variable_pc_details <- function(analysis_results, variable_names) {
  for (var in variable_names) {
    cat("\n===== Variable:", var, "=====\n")
    
    # Get correlations for this variable
    var_details <- get_variable_pc_details(analysis_results, var)
    
    if (nrow(var_details) > 0) {
      # Print the top 3 PC correlations
      top_pcs <- var_details %>% 
        arrange(p_adj) %>% 
        head(3)
      
      for (i in 1:nrow(top_pcs)) {
        cat("PC:", top_pcs$pc[i], 
            "| Correlation:", round(top_pcs$correlation[i], 3),
            "| p-value:", formatC(top_pcs$p_value[i], format = "e", digits = 2),
            "| Variance explained:", round(top_pcs$var_explained[i], 2), "%\n")
      }
    } else {
      cat("No correlation data found for this variable\n")
    }
  }
}

# Run the function for your identified covariates
print_variable_pc_details(initial_analysis, identified_covariates)
===== Variable: Lymphocyte_total_wilk =====
PC: PC4 | Correlation: 0.833 | p-value: 0.00e+00 | Variance explained: 6.91 %
PC: PC3 | Correlation: -0.701 | p-value: 0.00e+00 | Variance explained: 9.9 %
PC: PC1 | Correlation: -0.378 | p-value: 1.58e-09 | Variance explained: 19.94 %

===== Variable: RNASeqMetrics_MEDIAN_3PRIME_BIAS =====
PC: PC1 | Correlation: -0.634 | p-value: 0.00e+00 | Variance explained: 19.94 %
PC: PC6 | Correlation: -0.337 | p-value: 8.79e-08 | Variance explained: 4.22 %
PC: PC7 | Correlation: -0.282 | p-value: 8.77e-06 | Variance explained: 3 %

===== Variable: STAR_Number_of_splices_Annotated__sjdb_ =====
PC: PC7 | Correlation: 0.36 | p-value: 9.73e-09 | Variance explained: 3 %
PC: PC6 | Correlation: 0.313 | p-value: 7.49e-07 | Variance explained: 4.22 %
PC: PC8 | Correlation: -0.303 | p-value: 1.70e-06 | Variance explained: 2.05 %

===== Variable: STAR_Number_of_reads_unmapped_other =====
PC: PC1 | Correlation: 0.578 | p-value: 0.00e+00 | Variance explained: 19.94 %
PC: PC2 | Correlation: 0.478 | p-value: 0.00e+00 | Variance explained: 15.34 %
PC: PC6 | Correlation: 0.437 | p-value: 9.35e-13 | Variance explained: 4.22 %

===== Variable: RNASeqMetrics_PCT_INTRONIC_BASES =====
PC: PC5 | Correlation: -0.628 | p-value: 0.00e+00 | Variance explained: 5.44 %
PC: PC1 | Correlation: 0.575 | p-value: 0.00e+00 | Variance explained: 19.94 %
PC: PC6 | Correlation: 0.478 | p-value: 0.00e+00 | Variance explained: 4.22 %

## VISUALIZATION 
# Function to create the most informative visualization
plot_top_correlations <- function(analysis_results, variables) {
  # Extract correlations for the specified variables
  correlation_data <- analysis_results$correlations %>%
    filter(variable %in% variables) %>%
    # Add significance level
    mutate(
      significance = ifelse(p_adj < 0.01, "Significant", "Not significant"),
      pc_num = as.numeric(gsub("PC", "", pc))
    ) %>%
    # Keep only the first 5 PCs for clarity
    filter(pc_num <= 5) %>%
    # Arrange for consistent plotting
    arrange(variable, pc_num)
  
  # Create a clear heatmap with informative labels
  ggplot(correlation_data, aes(x = pc, y = variable, fill = correlation)) +
    geom_tile(color = "white") +
    # Add correlation values as text
    geom_text(aes(label = sprintf("%.2f", round(correlation, 2))), 
              color = ifelse(abs(correlation_data$correlation) > 0.5, "white", "black"),
              size = 3) +
    # Use a diverging color scale
    scale_fill_gradient2(low = "navy", mid = "white", high = "firebrick", 
                        midpoint = 0, limits = c(-1, 1)) +
    # Add variance explained to PC labels
    scale_x_discrete(
      labels = paste0(unique(correlation_data$pc), "\n(", 
                     round(correlation_data$var_explained[match(unique(correlation_data$pc), correlation_data$pc)], 1), "%)")
    ) +
    # Use shorter variable names if needed
    scale_y_discrete(labels = function(x) {
      sapply(x, function(i) {
        if(nchar(i) > 25) paste0(substr(i, 1, 22), "...") else i
      })
    }) +
    labs(title = "Correlation of Confounding Variables with Principal Components",
         subtitle = "Values show correlation coefficients, colors indicate direction and strength",
         x = "Principal Component (% variance explained)", 
         y = "Confounding Variable",
         fill = "Correlation") +
    theme_minimal() +
    theme(
      plot.title = element_text(size = 11, face = "bold"),
      plot.subtitle = element_text(size = 9),
      axis.text.y = element_text(size = 9),
      axis.text.x = element_text(size = 9, face = "bold")
    )
}

# Call the function with your data
plot_top_correlations(initial_analysis, identified_covariates)

In [ ]:
# PLOT ALL 10 PCs
# Function to create visualization for all 10 PCs
plot_all_pc_correlations <- function(analysis_results, variables) {
  # Extract correlations for the specified variables
  correlation_data <- analysis_results$correlations %>%
    filter(variable %in% variables) %>%
    # Add significance level
    mutate(
      significance = ifelse(p_adj < 0.01, "Significant", "Not significant"),
      pc_num = as.numeric(gsub("PC", "", pc))
    ) %>%
    # Keep only PCs 1-10
    filter(pc_num <= 10) %>%
    # Arrange for consistent plotting
    arrange(variable, pc_num)
  
  # Create a clear heatmap with informative labels
  ggplot(correlation_data, aes(x = pc, y = variable, fill = correlation)) +
    geom_tile(color = "white") +
    # Add correlation values as text
    geom_text(aes(label = sprintf("%.2f", round(correlation, 2))), 
              color = ifelse(abs(correlation_data$correlation) > 0.5, "white", "black"),
              size = 3) +
    # Use a diverging color scale
    scale_fill_gradient2(low = "navy", mid = "white", high = "firebrick", 
                        midpoint = 0, limits = c(-1, 1)) +
    # Add variance explained to PC labels
    scale_x_discrete(
      labels = paste0(unique(correlation_data$pc), "\n(", 
                     round(correlation_data$var_explained[match(unique(correlation_data$pc), correlation_data$pc)], 1), "%)")
    ) +
    # Add stars for significance
    geom_text(data = subset(correlation_data, p_adj < 0.05),
              aes(label = ifelse(p_adj < 0.001, "***", 
                               ifelse(p_adj < 0.01, "**", "*"))),
              vjust = -0.7, size = 3) +
    labs(title = "Correlation of Confounding Variables with Principal Components",
         subtitle = "Values show correlation coefficients; * p<0.05, ** p<0.01, *** p<0.001",
         x = "Principal Component (% variance explained)", 
         y = "Confounding Variable",
         fill = "Correlation") +
    theme_minimal() +
    theme(
      plot.title = element_text(size = 11, face = "bold"),
      plot.subtitle = element_text(size = 9),
      axis.text.y = element_text(size = 9),
      axis.text.x = element_text(size = 9),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank()
    )
}

# Call the function with your data
plot_all_pc_correlations(initial_analysis, identified_covariates)

In [ ]:
####### BLOOD
# Convert raw counts to DGEList
dge_blood <- DGEList(counts = blood_ge) #creates 2 lists, dge_blood$counts &  dge_blood$samples 
#this propoerly tracks library sizes, required for downstream calcNormFactors, and keep counts and sample info tgt

# Convert to CPM
cpm_blood <- cpm(dge_blood)

# Filtering: Keep genes with CPM >= 1 in at least 10% of samples
min_samples <- ceiling(0.1 * ncol(blood_ge))  # 10% of samples
keep_genes <- rowSums(cpm_blood >= 1) >= min_samples
dge_blood <- dge_blood[keep_genes, , keep.lib.sizes = FALSE]  # Apply filtering

# VPA
form <- ~ mymet_rin + (1|IID_ISMMS) + (1|mymet_sex)
varPart <- fitExtractVarPartModel(vobj_blood, form, blood_metadata)
plot <- plotVarPart(varPart) + theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Proportion of Variance Explained by Each Covariate")

#ggsave(filename = "/hpc/users/hoangd02/www/plots/lbp/blood_variance_partition_all_samples.png",
#       plot = plot)

# OPTION 1: Apply TMM normalization for composition bias -- this is one way
voomWithDreamWeights is prefered as it include random effect dynamically
dge_blood <- calcNormFactors(dge_blood, method = "TMM")
vobj_blood <- voom(dge_blood, plot = TRUE)
fit <- dream(vobj_blood, ~ mymet_rin + (1|IID_ISMMS) + (1|mymet_sex), blood_metadata)

# Get residuals
blood_resid_matrix <- residuals(fit)
dim(blood_resid_matrix)
#21296   265

write.table(blood_resid_matrix, 
            file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_resid_matrix_all_samples.txt", 
            sep = "\t",            # tab-delimited
            quote = FALSE,         # don't put quotes around values
            row.names = TRUE,      # keep gene names as rownames
            col.names = NA)        # keep sample names as column headers


##PCA on residualized matrix
# Center the residual matrix (genes x samples)
# If rows = genes, columns = samples
resid_matrix_t <- t(blood_resid_matrix)  # prcomp expects samples as rows

# Perform PCA
pca_res <- prcomp(resid_matrix_t, center = TRUE, scale. = TRUE)

# View variance explained
summary(pca_res)

pca_summary <- summary(pca_res)$importance
write.csv(pca_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_all_samples.csv", row.names = TRUE)
write.table(pca_summary, file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_all_samples.txt", sep = "\t", row.names = TRUE)

test<-fread("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/blood_pca_summary_all_samples.txt",data.table=FALSE)

#Importance of components:
#                           PC1     PC2     PC3      PC4      PC5     PC6
#Standard deviation     61.4137 52.7810 48.5841 39.05700 36.12381 32.1709
#Proportion of Variance  0.1782  0.1316  0.1115  0.07208  0.06166  0.0489
#Cumulative Proportion   0.1782  0.3099  0.4214  0.49347  0.55513  0.6040
#                            PC7      PC8      PC9     PC10     PC11     PC12
#Standard deviation     26.41472 20.90169 18.88612 17.64587 17.22997 14.87964
#Proportion of Variance  0.03297  0.02064  0.01685  0.01471  0.01403  0.01046
#Cumulative Proportion   0.63701  0.65765  0.67450  0.68922  0.70325  0.71371
#                           PC13     PC14     PC15     PC16     PC17    PC18
#Standard deviation     13.63565 13.19159 12.40876 11.70235 11.37328 11.1780
#Proportion of Variance  0.00879  0.00822  0.00728  0.00647  0.00611  0.0059
#Cumulative Proportion   0.72249  0.73072  0.73799  0.74446  0.75058  0.7565
#                           PC19     PC20     PC21   PC22    PC23    PC24   PC25
#Standard deviation     10.61020 10.26858 10.03675 9.7592 9.41204 9.36418 9.2000
#Proportion of Variance  0.00532  0.00498  0.00476 0.0045 0.00419 0.00414 0.0040
#Cumulative Proportion   0.76180  0.76678  0.77154 0.7760 0.78023 0.78437 0.7884
#                          PC26    PC27    PC28    PC29    PC30    PC31    PC32
#Standard deviation     9.13970 8.89706 8.62332 8.45369 8.24606 8.16679 8.03913
#Proportion of Variance 0.00395 0.00374 0.00351 0.00338 0.00321 0.00315 0.00305
#Cumulative Proportion  0.79232 0.79606 0.79957 0.80295 0.80616 0.80931 0.81237

## END HERE



In [ ]:
######## SKIP 
#Step 5: Network Analysis with WGCNA - do this for brain and blood
library(WGCNA)

# Transpose residual matrix so samples are rows
datExpr <- as.data.frame(t(resid_matrix))

# Ensure good samples and genes
gsg <- goodSamplesGenes(datExpr, verbose = 3)
datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]

# Choose soft threshold
powers <- c(1:20)
sft <- pickSoftThreshold(datExpr, powerVector = powers, verbose = 5)
softPower <- sft$powerEstimate

# Build network
net <- blockwiseModules(datExpr, power = softPower, TOMType = "unsigned",
                        minModuleSize = 30, reassignThreshold = 0,
                        mergeCutHeight = 0.25, numericLabels = TRUE,
                        pamRespectsDendro = FALSE, verbose = 3)

# Extract module eigengenes
MEs <- net$MEs

#✅ Step 6: Correlate Covariates to Module Eigengenes

# Combine metadata with module eigengenes
mod_df <- cbind(merged_metadata, MEs)

# Canonical correlation again
cc_wgcna <- canCorPairs(mod_df)

# Plot or review
plot(cc_wgcna)


In [ ]:
# Distribution of log2_cpm in BRAIN
# Flatten the matrix/dataframe into a long vector
brain_log2_cpm_values <- as.vector(as.matrix(brain_log2_cpm))

# Plot histogram of all log2CPM values using updated syntax
plot <- ggplot(data.frame(log2CPM = brain_log2_cpm_values), aes(x = log2CPM)) +
  geom_histogram(aes(y = after_stat(count) * 100 / sum(after_stat(count))),
                 binwidth = 0.25, boundary = 0) +
  xlab("Brain log2(CPM)") +
  ylab("Percent of values in bin") +
  ggtitle("Distribution of log2 CPM values in brain samples") +
  theme_minimal()

ggsave(filename = "/hpc/users/hoangd02/www/plots/brain_log2_cpm.png",
       plot = plot, 
       width = 6, height = 6, dpi = 300)

https://hoangd02.u.hpc.mssm.edu/plots/brain_log2_cpm.png

# Distribution of log2_cpm in BLOOD
blood_log2_cpm_values <- as.vector(as.matrix(blood_log2_cpm))

# Plot histogram of all log2CPM values using updated syntax
plot <- ggplot(data.frame(log2CPM = blood_log2_cpm_values), aes(x = log2CPM)) +
  geom_histogram(aes(y = after_stat(count) * 100 / sum(after_stat(count))),
                 binwidth = 0.25, boundary = 0) +
  xlab("Blood log2(CPM)") +
  ylab("Percent of values in bin") +
  ggtitle("Distribution of log2 CPM values in blood samples") +
  theme_minimal()

ggsave(filename = "/hpc/users/hoangd02/www/plots/blood_log2_cpm.png",
       plot = plot, 
       width = 6, height = 6, dpi = 300)

https://hoangd02.u.hpc.mssm.edu/plots/blood_log2_cpm.png

#after_stat(count) = the number of observations (e.g., genes) that fall into each histogram bin.
#sum(after_stat(count)) = the total number of observations across all bins.

#after_stat(count) * 100 / sum(after_stat(count)) = the percent of observations in each bin.

### There are TECHNICAL and BIOLOGICAL SOURCES of VARIATION 

#### TECHNICAL 
##### (1) KNOWN
- ComBat, and linear model-based differential expression methods, such as edgeR, limma, and DESeq2 can make a direct adjustment for known sources of unwanted variation only.
##### (2) UNKNOWN
- SVA, Seurat, and probabilistic estimation of expression residuals factors are generally agnostic to the cause of sample heterogeneity and create cova- riates that regress out/correct for variation without regard to the reason for heterogeneity.

#### BIOLOGICAL 
Age, sex, but some variables, including the manner of death and medication usage, or cellular composition differences of tissues due to aging or disease-related processes (e.g., atrophy and inflammation) or in variable harvesting can lead to different patterns of tissue gene expression....


### Notes and Resources

#### Notes
(1) For preliminary QC of paired-samples, look at blood-brain LBP_clean.sh

#### Resources
(1) Ryan's QC: Exploration of CD4 RNA-Seq Dataset https://darwinawardwinner.github.io/resume/examples/Salomon/CD4/reports/RNA-seq/salmon_hg38.analysisSet_ensembl.85-exploration.html

(2) Ryan's MOFA: MOFA analysis of RNA-seq and promoter histone ChIP-seq data
https://darwinawardwinner.github.io/resume/examples/Salomon/CD4/reports/MOFA/promoter-mofa-analyze.html
